# Structured R1 — Semantic Policy Refinement

This notebook follows the controlled Structured R1 branch-ablation study.

The preceding participation, local-temporal, global-temporal, and semantic ablations established a strong predictive configuration, but they also exposed an important diagnostic problem: **semantic evidence was being under-utilised inside the unified reasoner**.

The selected evidence configuration entering this experiment is kept fixed:

```text
Participation:
    `speaks` only
    (`filtered_turns` removed)

Local temporal:
    response-offset distribution only
    (overlap removed)

Global temporal:
    all five global features retained

Semantic:
    coarse + focused summaries
```

No evidence feature is added or removed in this notebook.

Instead, the intervention is entirely at the **prompt-policy level**.

The objective is to make the Structured R1 `semantic_assessment` behave as an independent semantic diagnostic rather than allowing temporal or participation evidence to influence whether two participant records are judged semantically compatible.

---

# Why revise the semantic policy?

The branch ablations showed that the unified reasoner could achieve high predictive accuracy while still using semantic evidence weakly.

This was especially visible for Wrong Partner cases: many cases were correctly classified as anomalous, but the structured output often did **not** identify the participant content itself as `INCOMPATIBLE`.

That behaviour suggested that Wrong Partner detection was frequently being carried by temporal evidence rather than by the semantic branch that should naturally detect conversational mismatch.

The goal of this experiment is therefore not simply to maximise classification accuracy.

It is to improve **semantic fidelity**:

```text
Wrong Partner
        ↓
semantic mismatch should be recognised
        ↓
semantic_assessment = INCOMPATIBLE
```

while preserving the existing participation and temporal evidence pipeline.

---

# What changes in the prompt?

The complete selected Structured R1 prompt is loaded as the baseline.

The intervention is deliberately surgical:

```text
Everything before SEMANTIC EVIDENCE      → unchanged
SEMANTIC EVIDENCE policy                 → replaced
FINAL COMBINED DECISION and everything
after it                                 → unchanged
```

The notebook explicitly verifies these prompt boundaries and saves a prompt diff.

Only the semantic-policy section is rewritten.

## 1. Semantic assessment becomes branch-independent

The revised prompt explicitly states that `semantic_assessment` must be determined **only from the supplied semantic summaries**.

When assigning the semantic assessment, the model is instructed not to use:

- participation evidence,
- local temporal evidence,
- global temporal evidence,
- whether the temporal pattern appears NORMAL or ANOMALOUS,
- or another branch to soften or strengthen the semantic judgement.

Conceptually:

```text
semantic_assessment
        =
semantic evidence only
```

This directly addresses the cross-branch coupling observed in the preceding experiments.

---

## 2. Coarse and focused summaries are interpreted together

For each synchronized 60-second segment, the reasoner receives two independently generated descriptions of the same participant content:

### Coarse summary
- `speech_content_summary`
- `apparent_topic`

### Focused summary
- `detailed_speech_summary`
- `main_topic`
- `secondary_topics`
- `key_semantic_details`
- `summary_specificity`
- `unclear_content`
- `confidence`

The revised policy tells the model to interpret the coarse and focused summaries **jointly**.

Differences between one participant's own coarse and focused descriptions must not themselves be interpreted as evidence of mismatch, because the two summaries are independently generated views of the same segment.

---

## 3. Semantic comparison is explicitly participant-to-participant

The semantic branch must independently answer:

> Can the two participants' supplied content plausibly belong to the same real 120-second dyadic conversation?

The prompt instructs the model to internally inspect:

1. Participant A vs Participant B in Segment 0;
2. Participant A vs Participant B in Segment 1;
3. the complete cross-segment relationship across the full 120 seconds.

These checks are internal and do not create additional output fields.

The model is also explicitly told **not to use a simple vote between the two segments**.

---

## 4. Concrete unrelated content is `INCOMPATIBLE`

A major change concerns the distinction between mismatch and uncertainty.

The revised policy makes the following rule explicit:

```text
specific + concrete + unrelated content
        →
INCOMPATIBLE
```

It must **not** be downgraded to `LIMITED` merely because the model cannot invent a plausible connection.

Examples of strong incompatibility include:

- repeatedly unrelated concrete subjects;
- incompatible people, events, places, activities, products, stories, situations, or arguments;
- one participant discussing a concrete subject while the other discusses a clearly unrelated concrete subject;
- both synchronized segments showing mismatch;
- one strong concrete mismatch while the other segment provides no credible compatibility evidence.

The model is explicitly instructed **not to invent indirect relationships merely to make two participant records compatible**.

---

## 5. Broad thematic similarity is not enough for `COMPATIBLE`

The revised prompt prevents generic category matching from being treated as evidence of a shared conversation.

For example, two summaries are not automatically compatible merely because both concern broad categories such as:

- personal experiences,
- preferences,
- opinions,
- daily life,
- family,
- relationships,
- health,
- lifestyle,
- or personal well-being.

`COMPATIBLE` now requires **concrete evidence of a plausible shared conversational relationship**, such as:

- a shared subject,
- compatible people/events/places,
- complementary accounts,
- question-response structure,
- reaction or elaboration,
- different aspects of the same subject,
- or a coherent topic transition.

---

## 6. `LIMITED` is reserved for genuinely insufficient semantic evidence

The revised policy distinguishes uncertainty from incompatibility.

```text
specific but unrelated
        → INCOMPATIBLE

vague / generic / unclear / incomplete
        → potentially LIMITED
```

`LIMITED` should be used only when the semantic evidence is genuinely too weak or uncertain to support either compatibility or incompatibility.

A vague segment does not automatically make the complete 120-second assessment `LIMITED`; the remaining segment and the full interaction pattern must still be considered.

---

## 7. Semantic and temporal evidence cannot cancel each other

The revised policy makes the branch relationship explicit:

```text
Semantic COMPATIBLE
    must not cancel reliable temporal failure.

Semantic INCOMPATIBLE
    must not be cancelled by apparently normal temporal coordination.

Semantic LIMITED
    is inconclusive and does not itself establish NORMAL or ANOMALOUS.
```

The semantic field is therefore computed as an **independent diagnostic assessment**, after which the unchanged final Structured R1 decision policy combines the branches.

---

# Controlled Experimental Setup

Everything outside the semantic policy remains fixed:

- the same 400 development cases;
- the same Qwen2.5-Omni model;
- the same deterministic decoding settings;
- `speaks` only for participation;
- no `filtered_turns`;
- complete response-offset distribution;
- no overlap features;
- all five global temporal features;
- the same coarse summaries;
- the same focused summaries;
- the same frozen NORMAL reference;
- the same Structured R1 output schema;
- the same final combined-decision policy.

The notebook also prints:

1. the targeted Normal-vs-Wrong-Partner semantic prompt that motivated the intervention;
2. the exact selected Structured R1 baseline prompt;
3. the complete manually defined revised prompt;
4. the exact prompt diff;
5. a rendered case-level prompt for final inspection before inference.

This ensures that the experiment is a controlled **semantic-policy intervention**, not an untracked prompt rewrite.

---

# Result

The revised semantic policy produces:

| Configuration | NORMAL | LAG | WRONG | SILENT | Accuracy | WP `INCOMPATIBLE` |
|---|---:|---:|---:|---:|---:|---:|
| **Revised semantic policy** | **84/100** | **74/100** | **90/100** | **100/100** | **87.00%** | **40/100** |

The case-family prediction table is:

```text
NORMAL          84 / 100
LAG             74 / 100
WRONG PARTNER   90 / 100
SILENT PARTNER 100 / 100
```

More importantly, the Wrong Partner semantic-assessment distribution now contains:

```text
COMPATIBLE      45
INCOMPATIBLE    40
LIMITED         15
```

The key improvement is therefore not raw predictive accuracy alone.

The revised policy substantially increases the number of Wrong Partner cases that the reasoner explicitly recognises as **semantically incompatible**, demonstrating that the semantic branch is now more aligned with its intended function.

This revised policy is consequently selected as the semantic-policy baseline for the next stage: **semantic representation ablation**, where the semantic inputs themselves are varied while this decision policy is kept fixed.

> **Reproducibility note:** every code cell, saved output, execution count, code-cell metadata field, manually defined prompt, prompt inspection, evaluation table, and diagnostic output is preserved exactly as executed. Only this introductory Markdown cell has been rewritten for the public repository.

## 1. Install dependencies

Run this cell once in a fresh Colab runtime. Then select:

`Runtime → Restart session`

and continue from the next cell.

In [ ]:
!pip uninstall -y transformers
!pip install -U "transformers>=4.57.0" accelerate bitsandbytes sentencepiece
!pip install -U qwen-omni-utils decord ffmpeg-python
!apt-get update -qq
!apt-get install -y -qq ffmpeg

Found existing installation: transformers 5.13.1
Uninstalling transformers-5.13.1:
  Successfully uninstalled transformers-5.13.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 146.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 62.4 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


## 2. Mount Google Drive and configure artifact paths

The folder name still contains `1_2_3sec` for historical reasons. The finalized 400-case database is loaded from the exact path below.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")


from pathlib import Path
from collections import Counter
import copy
import json

import pandas as pd
from IPython.display import display


OUT_DIR = Path(
    "/content/drive/MyDrive/"
    "qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec"
)


REFERENCE_BASE_STATS_PATH = (
    OUT_DIR
    / "frozen_reference_base_statistics.json"
)


REFERENCE_SHIFT_STATS_PATH = (
    OUT_DIR
    / "frozen_reference_global_shift_statistics.json"
)


FINAL_DATABASE_PATH = (
    OUT_DIR
    / (
        "consolidation_all_400_cases_with_"
        "temporal_and_semantic_summaries.json"
    )
)


MODEL_ID = "Qwen/Qwen2.5-Omni-7B"

MODEL_PATH = Path(
    "/content/drive/MyDrive/Qwen2.5-Omni-7B"
)


required_paths = {
    "Frozen base statistics": (
        REFERENCE_BASE_STATS_PATH
    ),

    "Frozen global-shift statistics": (
        REFERENCE_SHIFT_STATS_PATH
    ),

    "Final 400-case database": (
        FINAL_DATABASE_PATH
    ),

    "Local Qwen checkpoint": (
        MODEL_PATH
    ),
}


print("=" * 88)
print("ARTIFACT PATH AUDIT")
print("=" * 88)

for name, path in required_paths.items():

    print(
        f"{name}:",
        path,
    )

    print(
        "  exists:",
        path.exists(),
    )


assert OUT_DIR.exists(), (
    f"Project directory not found: {OUT_DIR}"
)

assert REFERENCE_BASE_STATS_PATH.exists(), (
    "Frozen base-statistics file not found: "
    f"{REFERENCE_BASE_STATS_PATH}"
)

assert REFERENCE_SHIFT_STATS_PATH.exists(), (
    "Frozen global-shift-statistics file not found: "
    f"{REFERENCE_SHIFT_STATS_PATH}"
)

assert FINAL_DATABASE_PATH.exists(), (
    "Final consolidation database not found: "
    f"{FINAL_DATABASE_PATH}"
)

assert MODEL_PATH.exists(), (
    "Local Qwen checkpoint not found: "
    f"{MODEL_PATH}"
)

Mounted at /content/drive
ARTIFACT PATH AUDIT
Frozen base statistics: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/frozen_reference_base_statistics.json
  exists: True
Frozen global-shift statistics: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/frozen_reference_global_shift_statistics.json
  exists: True
Final 400-case database: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_all_400_cases_with_temporal_and_semantic_summaries.json
  exists: True
Local Qwen checkpoint: /content/drive/MyDrive/Qwen2.5-Omni-7B
  exists: True


## 3. Load the frozen NORMAL reference statistics

Both frozen JSON files contain separate reference profiles. This experiment deliberately extracts and retains only:

```text
reference_profile = NORMAL
```

The lag reference profiles are not included in the binary-only experiment state at this stage.

In [ ]:
# ============================================================
# LOAD FROZEN STATISTICS AND RETAIN NORMAL ONLY
# ============================================================

def load_json_list(
    path,
):
    data = json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


    assert isinstance(
        data,
        list,
    ), (
        f"Expected a JSON list in {path}"
    )


    return data


frozen_base_records_all_profiles = (
    load_json_list(
        REFERENCE_BASE_STATS_PATH
    )
)


frozen_shift_records_all_profiles = (
    load_json_list(
        REFERENCE_SHIFT_STATS_PATH
    )
)


base_profiles = [
    str(
        record.get(
            "reference_profile"
        )
    )

    for record
    in frozen_base_records_all_profiles
]


shift_profiles = [
    str(
        record.get(
            "reference_profile"
        )
    )

    for record
    in frozen_shift_records_all_profiles
]


assert len(
    base_profiles
) == len(
    set(
        base_profiles
    )
)


assert len(
    shift_profiles
) == len(
    set(
        shift_profiles
    )
)


assert "NORMAL" in base_profiles
assert "NORMAL" in shift_profiles


normal_base_matches = [
    record

    for record
    in frozen_base_records_all_profiles

    if str(
        record.get(
            "reference_profile"
        )
    ) == "NORMAL"
]


normal_shift_matches = [
    record

    for record
    in frozen_shift_records_all_profiles

    if str(
        record.get(
            "reference_profile"
        )
    ) == "NORMAL"
]


assert len(
    normal_base_matches
) == 1


assert len(
    normal_shift_matches
) == 1


frozen_normal_base_statistics = copy.deepcopy(
    normal_base_matches[0]
)


frozen_normal_shift_statistics = copy.deepcopy(
    normal_shift_matches[0]
)


assert (
    frozen_normal_base_statistics[
        "reference_profile"
    ]
    == "NORMAL"
)


assert (
    frozen_normal_shift_statistics[
        "reference_profile"
    ]
    == "NORMAL"
)


# This is the only frozen reference object that should later
# be passed to the binary-only prompt constructor.
FROZEN_NORMAL_REFERENCE = {
    "base_statistics": (
        frozen_normal_base_statistics
    ),

    "global_shift_statistics": (
        frozen_normal_shift_statistics
    ),
}


normal_base_display_df = pd.DataFrame([
    {
        "metric": key,
        "value": value,
    }

    for key, value
    in frozen_normal_base_statistics.items()

    if key != "reference_profile"
])


normal_shift_display_df = pd.DataFrame([
    {
        "metric": key,
        "value": value,
    }

    for key, value
    in frozen_normal_shift_statistics.items()

    if key != "reference_profile"
])


print("=" * 88)
print("FROZEN REFERENCE FILES LOADED")
print("=" * 88)

print(
    "Profiles stored in base-statistics file:",
    base_profiles,
)

print(
    "Profiles stored in global-shift file:",
    shift_profiles,
)


print("\n" + "=" * 88)
print("FROZEN NORMAL BASE TEMPORAL STATISTICS")
print("=" * 88)

display(
    normal_base_display_df
)


print("\n" + "=" * 88)
print("FROZEN NORMAL GLOBAL-SHIFT STATISTICS")
print("=" * 88)

display(
    normal_shift_display_df
)


print(
    "\nOnly NORMAL frozen references retained:",
    list(
        FROZEN_NORMAL_REFERENCE.keys()
    ),
)

FROZEN REFERENCE FILES LOADED
Profiles stored in base-statistics file: ['NORMAL', 'LAG_1', 'LAG_2', 'LAG_3']
Profiles stored in global-shift file: ['NORMAL', 'LAG_1', 'LAG_2', 'LAG_3']

FROZEN NORMAL BASE TEMPORAL STATISTICS


,metric,value
0,num_original_A_turns,11.800000
1,num_original_B_turns,11.080000
2,num_removed_A_backchannels,1.740000
3,num_removed_B_backchannels,1.660000
4,num_filtered_A_turns,10.060000
5,num_filtered_B_turns,9.420000
6,num_offsets,5.760000
7,num_negative,2.040000
8,num_positive,3.720000
9,offset_mean,0.302128



FROZEN NORMAL GLOBAL-SHIFT STATISTICS


,metric,value
0,best_B_correction_shift_seconds__mean,-0.230000
1,best_B_correction_shift_seconds__median,-0.000000
2,estimated_B_lateness_seconds__mean,0.500000
3,estimated_B_lateness_seconds__median,0.000000
4,alignment_score_gain_vs_zero__mean,0.023376
5,alignment_score_gain_vs_zero__median,0.009105
6,best_num_bilateral_events__mean,11.940000
7,best_event_coverage__mean,0.595175



Only NORMAL frozen references retained: ['base_statistics', 'global_shift_statistics']


## 4. Load and audit the final 400-case database

This cell verifies:

- exactly 400 unique cases;
- exactly 100 cases per family;
- 100 `NORMAL` and 300 `ANOMALOUS` binary labels;
- participant-level `speaks` agrees with the final filtered VAD turns;
- every sample has all four coarse and all four focused summaries;
- no focused summary contains the legacy `speaks` field.

In [ ]:
# ============================================================
# LOAD FINAL DATABASE
# ============================================================

consolidation_cases = load_json_list(
    FINAL_DATABASE_PATH
)


assert len(
    consolidation_cases
) == 400


# ============================================================
# NORMALIZE CASE FAMILY
# ============================================================

def get_case_family(
    case,
):
    variant = str(
        case.get(
            "case_variant",
            "",
        )
    ).lower()


    if variant == "normal":
        return "normal"


    if variant == "wrong_partner":
        return "wrong_partner"


    if variant == "silent_partner":
        return "silent_partner"


    if variant.startswith(
        "lag"
    ):
        return "lag"


    raise ValueError(
        "Unknown case_variant: "
        f"{case.get('case_variant')}"
    )


# ============================================================
# GLOBAL COUNTS
# ============================================================

case_ids = [
    str(
        case[
            "case_id"
        ]
    )

    for case
    in consolidation_cases
]


assert len(
    case_ids
) == len(
    set(
        case_ids
    )
)


family_counts = Counter(
    get_case_family(
        case
    )

    for case
    in consolidation_cases
)


expected_family_counts = {
    "normal": 100,
    "wrong_partner": 100,
    "lag": 100,
    "silent_partner": 100,
}


assert dict(
    family_counts
) == expected_family_counts


binary_label_counts = Counter(
    str(
        case[
            "gold_binary_label"
        ]
    )

    for case
    in consolidation_cases
)


assert binary_label_counts[
    "NORMAL"
] == 100


assert binary_label_counts[
    "ANOMALOUS"
] == 300


lag_variant_counts = Counter(
    str(
        case.get(
            "case_variant"
        )
    )

    for case
    in consolidation_cases

    if get_case_family(
        case
    ) == "lag"
)


# ============================================================
# PARTICIPANT AND SEMANTIC AUDITS
# ============================================================

SEGMENT_NAMES = [
    "segment_0_0_to_60_seconds",
    "segment_1_60_to_120_seconds",
]


num_participant_records_checked = 0
num_semantic_slots_checked = 0


for case in consolidation_cases:

    for (
        role,
        turns_key,
    ) in [
        (
            "participant_A",
            "participant_A_filtered_turns",
        ),

        (
            "participant_B",
            "participant_B_filtered_turns",
        ),
    ]:

        participant = case[
            role
        ]


        turns = case[
            turns_key
        ]


        assert isinstance(
            participant,
            dict,
        )


        assert isinstance(
            turns,
            list,
        )


        assert (
            "speaks"
            in participant
        )


        assert isinstance(
            participant[
                "speaks"
            ],
            bool,
        )


        assert (
            participant[
                "speaks"
            ]
            ==
            (
                len(
                    turns
                ) > 0
            )
        ), (
            "Participant-level speaks disagrees with "
            f'VAD turns in {case["case_id"]} / {role}'
        )


        participant_semantics = (
            case[
                "semantic_summaries"
            ][role]
        )


        for segment_name in SEGMENT_NAMES:

            segment_record = (
                participant_semantics[
                    segment_name
                ]
            )


            coarse_summary = (
                segment_record.get(
                    "coarse_summary"
                )
            )


            focused_summary = (
                segment_record.get(
                    "focused_summary"
                )
            )


            assert isinstance(
                coarse_summary,
                dict,
            ), (
                "Missing coarse summary in "
                f'{case["case_id"]} / {role} / {segment_name}'
            )


            assert isinstance(
                focused_summary,
                dict,
            ), (
                "Missing focused summary in "
                f'{case["case_id"]} / {role} / {segment_name}'
            )


            assert (
                "speaks"
                not in focused_summary
            ), (
                "Legacy focused-summary speaks field found in "
                f'{case["case_id"]} / {role} / {segment_name}'
            )


            num_semantic_slots_checked += 1


        num_participant_records_checked += 1


# ============================================================
# DISPLAY DATABASE COMPOSITION
# ============================================================

family_count_df = pd.DataFrame([
    {
        "case_family": family,
        "num_cases": count,
    }

    for family, count
    in sorted(
        family_counts.items()
    )
])


binary_count_df = pd.DataFrame([
    {
        "gold_binary_label": label,
        "num_cases": count,
    }

    for label, count
    in sorted(
        binary_label_counts.items()
    )
])


lag_variant_df = pd.DataFrame([
    {
        "case_variant": variant,
        "num_cases": count,
    }

    for variant, count
    in sorted(
        lag_variant_counts.items()
    )
])


print("=" * 88)
print("FINAL CONSOLIDATION DATABASE AUDIT PASSED")
print("=" * 88)

print(
    "Database:",
    FINAL_DATABASE_PATH,
)

print(
    "Total unique cases:",
    len(
        consolidation_cases
    ),
)

print(
    "Participant records checked:",
    num_participant_records_checked,
)

print(
    "Semantic participant-segment slots checked:",
    num_semantic_slots_checked,
)


print("\nCASE FAMILIES")

display(
    family_count_df
)


print("\nBINARY LABELS")

display(
    binary_count_df
)


print("\nLAG VARIANTS STORED IN THE FINAL DATABASE")

display(
    lag_variant_df
)

FINAL CONSOLIDATION DATABASE AUDIT PASSED
Database: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_all_400_cases_with_temporal_and_semantic_summaries.json
Total unique cases: 400
Participant records checked: 800
Semantic participant-segment slots checked: 1600

CASE FAMILIES


,case_family,num_cases
0,lag,100
1,normal,100
2,silent_partner,100
3,wrong_partner,100



BINARY LABELS


,gold_binary_label,num_cases
0,ANOMALOUS,300
1,NORMAL,100



LAG VARIANTS STORED IN THE FINAL DATABASE


,case_variant,num_cases
0,lag_2sec,50
1,lag_3sec,50


## 5. Participant-level `speaks` overview

The field is derived exclusively from each sample's final filtered VAD turns.

In [ ]:
# ============================================================
# PARTICIPANT-LEVEL SPEAKS SUMMARY
# ============================================================

speaks_rows = []


for case in consolidation_cases:

    speaks_rows.append({
        "case_family": (
            get_case_family(
                case
            )
        ),

        "case_id": (
            case[
                "case_id"
            ]
        ),

        "participant_A_speaks": (
            case[
                "participant_A"
            ][
                "speaks"
            ]
        ),

        "participant_B_speaks": (
            case[
                "participant_B"
            ][
                "speaks"
            ]
        ),

        "participant_A_num_turns": len(
            case[
                "participant_A_filtered_turns"
            ]
        ),

        "participant_B_num_turns": len(
            case[
                "participant_B_filtered_turns"
            ]
        ),
    })


speaks_df = pd.DataFrame(
    speaks_rows
)


speaks_summary_df = (
    speaks_df
    .groupby(
        "case_family",
        as_index=False,
    )
    .agg(
        total_cases=(
            "case_id",
            "size",
        ),

        participant_A_speaks_true=(
            "participant_A_speaks",
            "sum",
        ),

        participant_B_speaks_true=(
            "participant_B_speaks",
            "sum",
        ),
    )
)


speaks_summary_df[
    "participant_A_speaks_false"
] = (
    speaks_summary_df[
        "total_cases"
    ]
    -
    speaks_summary_df[
        "participant_A_speaks_true"
    ]
)


speaks_summary_df[
    "participant_B_speaks_false"
] = (
    speaks_summary_df[
        "total_cases"
    ]
    -
    speaks_summary_df[
        "participant_B_speaks_true"
    ]
)


speaks_summary_df = speaks_summary_df[
    [
        "case_family",
        "total_cases",

        "participant_A_speaks_true",
        "participant_A_speaks_false",

        "participant_B_speaks_true",
        "participant_B_speaks_false",
    ]
]


display(
    speaks_summary_df
)

,case_family,total_cases,participant_A_speaks_true,participant_A_speaks_false,participant_B_speaks_true,participant_B_speaks_false
0,lag,100,100,0,100,0
1,normal,100,100,0,100,0
2,silent_partner,100,100,0,0,100
3,wrong_partner,100,100,0,100,0


## 6. Interactive case inspection

The inspector supports all four families:

- `NORMAL`
- `WRONG_PARTNER`
- `LAG`
- `SILENT_PARTNER`

It can display participant metadata, VAD-derived `speaks`, filtered turns, temporal features, semantic summaries, and the complete sample JSON.

The direct function can also be used without widgets:

```python
inspect_case(
    family="normal",
    sample_index=0,
)
```

In [ ]:
# ============================================================
# INTERACTIVE DATABASE INSPECTOR
# ============================================================

import ipywidgets as widgets

from IPython.display import (
    clear_output,
    display,
)


try:

    from google.colab import output

    output.enable_custom_widget_manager()

except Exception:

    pass


# ============================================================
# SPLIT AND SORT CASES
# ============================================================

cases_by_family = {
    "normal": [],
    "wrong_partner": [],
    "lag": [],
    "silent_partner": [],
}


for case in consolidation_cases:

    cases_by_family[
        get_case_family(
            case
        )
    ].append(
        case
    )


for family in cases_by_family:

    cases_by_family[
        family
    ] = sorted(
        cases_by_family[
            family
        ],

        key=lambda case: str(
            case.get(
                "case_id",
                "",
            )
        ),
    )


# ============================================================
# DISPLAY HELPERS
# ============================================================

def participant_row(
    case,
    role,
):
    participant = case[
        role
    ]


    turns_key = (
        f"{role}_filtered_turns"
    )


    turns = case.get(
        turns_key,
        [],
    )


    return {
        "role": role,

        "conversation_id": (
            participant.get(
                "conversation_id"
            )
        ),

        "participant_id": (
            participant.get(
                "participant_id"
            )
        ),

        "speaks": (
            participant.get(
                "speaks"
            )
        ),

        "num_filtered_turns": len(
            turns
        ),

        "metadata_path": (
            participant.get(
                "metadata_path"
            )
        ),
    }


def display_semantics(
    case,
    role,
):
    print(
        "\n" + "-" * 88
    )

    print(
        f"{role.upper()} SEMANTIC SUMMARIES"
    )

    print(
        "-" * 88
    )


    participant_semantics = (
        case[
            "semantic_summaries"
        ][role]
    )


    for segment_name in SEGMENT_NAMES:

        segment_record = (
            participant_semantics[
                segment_name
            ]
        )


        print(
            f"\nSEGMENT: {segment_name}"
        )


        print(
            "\nCoarse summary:"
        )


        print(
            json.dumps(
                segment_record[
                    "coarse_summary"
                ],
                indent=2,
                ensure_ascii=False,
            )
        )


        print(
            "\nFocused summary:"
        )


        print(
            json.dumps(
                segment_record[
                    "focused_summary"
                ],
                indent=2,
                ensure_ascii=False,
            )
        )


# ============================================================
# MAIN INSPECTION FUNCTION
# ============================================================

def inspect_case(
    family,
    sample_index,
    show_turns=True,
    show_temporal_features=True,
    show_semantics=True,
    show_full_json=False,
):
    assert family in cases_by_family


    family_cases = cases_by_family[
        family
    ]


    sample_index = int(
        sample_index
    )


    assert (
        0
        <= sample_index
        < len(
            family_cases
        )
    )


    case = family_cases[
        sample_index
    ]


    print("=" * 88)
    print("CONSOLIDATION CASE INSPECTION")
    print("=" * 88)

    print(
        "Family:",
        family,
    )

    print(
        "Family position:",
        f"{sample_index + 1}/{len(family_cases)}",
    )

    print(
        "case_id:",
        case.get(
            "case_id"
        ),
    )

    print(
        "source_group_id:",
        case.get(
            "source_group_id"
        ),
    )

    print(
        "case_variant:",
        case.get(
            "case_variant"
        ),
    )

    print(
        "gold_binary_label:",
        case.get(
            "gold_binary_label"
        ),
    )

    print(
        "gold_anomaly_type:",
        case.get(
            "gold_anomaly_type"
        ),
    )

    print(
        "A_source_conversation:",
        case.get(
            "A_source_conversation"
        ),
    )

    print(
        "B_source_conversation:",
        case.get(
            "B_source_conversation"
        ),
    )


    print(
        "\nPARTICIPANTS"
    )


    display(
        pd.DataFrame([
            participant_row(
                case,
                "participant_A",
            ),

            participant_row(
                case,
                "participant_B",
            ),
        ])
    )


    print(
        "\nSEMANTIC COVERAGE"
    )


    print(
        json.dumps(
            case.get(
                "semantic_summary_coverage"
            ),
            indent=2,
            ensure_ascii=False,
        )
    )


    if show_turns:

        print(
            "\n" + "=" * 88
        )

        print(
            "PARTICIPANT A FILTERED TURNS"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case.get(
                    "participant_A_filtered_turns"
                ),
                indent=2,
                ensure_ascii=False,
            )
        )


        print(
            "\n" + "=" * 88
        )

        print(
            "PARTICIPANT B FILTERED TURNS"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case.get(
                    "participant_B_filtered_turns"
                ),
                indent=2,
                ensure_ascii=False,
            )
        )


    if show_temporal_features:

        print(
            "\n" + "=" * 88
        )

        print(
            "LOCAL TEMPORAL FEATURES"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case.get(
                    "local_temporal_features"
                ),
                indent=2,
                ensure_ascii=False,
            )
        )


        print(
            "\n" + "=" * 88
        )

        print(
            "GLOBAL SHIFT FEATURES"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case.get(
                    "global_shift_features"
                ),
                indent=2,
                ensure_ascii=False,
            )
        )


    if show_semantics:

        display_semantics(
            case,
            "participant_A",
        )


        display_semantics(
            case,
            "participant_B",
        )


    if show_full_json:

        print(
            "\n" + "=" * 88
        )

        print(
            "COMPLETE SAMPLE JSON"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case,
                indent=2,
                ensure_ascii=False,
            )
        )


    return case


# ============================================================
# WIDGETS
# ============================================================

family_labels = {
    "normal": "NORMAL",
    "wrong_partner": "WRONG_PARTNER",
    "lag": "LAG",
    "silent_partner": "SILENT_PARTNER",
}


family_dropdown = widgets.Dropdown(
    options=[
        (
            family_labels[
                family
            ],
            family,
        )

        for family in [
            "normal",
            "wrong_partner",
            "lag",
            "silent_partner",
        ]
    ],

    value="normal",

    description="Family:",

    style={
        "description_width": "initial"
    },

    layout=widgets.Layout(
        width="450px"
    ),
)


case_dropdown = widgets.Dropdown(
    description="Sample:",

    style={
        "description_width": "initial"
    },

    layout=widgets.Layout(
        width="900px"
    ),
)


show_turns_checkbox = widgets.Checkbox(
    value=True,
    description="Show filtered turns",
    indent=False,
)


show_temporal_checkbox = widgets.Checkbox(
    value=True,
    description="Show temporal features",
    indent=False,
)


show_semantics_checkbox = widgets.Checkbox(
    value=True,
    description="Show semantic summaries",
    indent=False,
)


show_full_json_checkbox = widgets.Checkbox(
    value=False,
    description="Show complete JSON",
    indent=False,
)


inspector_output = widgets.Output()


def refresh_case_options(
    *args,
):
    family = family_dropdown.value


    family_cases = cases_by_family[
        family
    ]


    case_dropdown.options = [
        (
            (
                f"{index + 1:03d}/100 | "
                f'{case.get("case_id")} | '
                f'{case.get("case_variant")}'
            ),

            index,
        )

        for index, case
        in enumerate(
            family_cases
        )
    ]


    case_dropdown.value = 0


def render_case(
    *args,
):
    if case_dropdown.value is None:
        return


    with inspector_output:

        clear_output(
            wait=True
        )


        inspect_case(
            family=(
                family_dropdown.value
            ),

            sample_index=(
                case_dropdown.value
            ),

            show_turns=(
                show_turns_checkbox.value
            ),

            show_temporal_features=(
                show_temporal_checkbox.value
            ),

            show_semantics=(
                show_semantics_checkbox.value
            ),

            show_full_json=(
                show_full_json_checkbox.value
            ),
        )


family_dropdown.observe(
    refresh_case_options,
    names="value",
)


family_dropdown.observe(
    render_case,
    names="value",
)


case_dropdown.observe(
    render_case,
    names="value",
)


show_turns_checkbox.observe(
    render_case,
    names="value",
)


show_temporal_checkbox.observe(
    render_case,
    names="value",
)


show_semantics_checkbox.observe(
    render_case,
    names="value",
)


show_full_json_checkbox.observe(
    render_case,
    names="value",
)


refresh_case_options()


controls = widgets.VBox([
    family_dropdown,
    case_dropdown,

    widgets.HBox([
        show_turns_checkbox,
        show_temporal_checkbox,
    ]),

    widgets.HBox([
        show_semantics_checkbox,
        show_full_json_checkbox,
    ]),
])


print("=" * 88)
print("INTERACTIVE INSPECTOR READY")
print("=" * 88)

for family in [
    "normal",
    "wrong_partner",
    "lag",
    "silent_partner",
]:

    print(
        family,
        len(
            cases_by_family[
                family
            ]
        ),
    )


display(
    controls,
    inspector_output,
)


render_case()


INTERACTIVE INSPECTOR READY
normal 100
wrong_partner 100
lag 100
silent_partner 100


Output()

# Ηelper Functions For Qwen Calls

In [ ]:
def extract_json_from_text(text: str):
    text = text.strip()

    # remove markdown fences
    text = re.sub(r"^```(?:json)?", "", text.strip(), flags=re.IGNORECASE)
    text = re.sub(r"```$", "", text.strip())

    try:
        return json.loads(text)
    except Exception:
        pass

    # find first balanced JSON object
    start = text.find("{")
    if start == -1:
        return {"parse_error": True, "raw_output": text}

    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                candidate = text[start:i+1]
                try:
                    return json.loads(candidate)
                except Exception:
                    return {
                        "parse_error": True,
                        "raw_output": text,
                        "json_candidate": candidate,
                    }

    return {"parse_error": True, "raw_output": text}

def qwen_video_text(video_path: str, prompt: str, max_new_tokens: int = 350):
    messages = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "You are Qwen, a virtual human developed by the Qwen Team, "
                        "Alibaba Group, capable of perceiving auditory and visual inputs, "
                        "as well as generating text and speech."
                    )
                }
            ],
        },
        {
            "role": "user",
            "content": [
                {"type": "video", "video": video_path, "fps": 7.0},
                {"type": "text", "text": prompt},
            ],
        },
    ]

    text = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False
    )

    audios, images, videos = process_mm_info(
        messages,
        use_audio_in_video=True
    )

    inputs = processor(
        text=text,
        audio=audios,
        images=images,
        videos=videos,
        return_tensors="pt",
        padding=True,
        use_audio_in_video=True,
    )

    inputs = inputs.to(model.device)

    print("Input tokens:", inputs["input_ids"].shape[1])

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    generated = processor.batch_decode(
        output_ids[:, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    return generated

def qwen_text_only(prompt: str, max_new_tokens: int = 700):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        processor_kwargs={
            "padding": True,
        },
    )

    inputs = {
        k: v.to(model.device) if hasattr(v, "to") else v
        for k, v in inputs.items()
    }

    input_token_count = int(
        inputs["input_ids"].shape[-1]
    )


    print(
        "Input tokens:",
        input_token_count,
    )


    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    generated = processor.batch_decode(
        output_ids[:, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    return generated


# Shared frozen NORMAL reference text

This is the same reference-text construction used by the original binary experiments. No anomaly-specific profile is added here; R2 and R3 load their exact saved prompts, which already contain the frozen LAG₂/LAG₃ profile text.

In [ ]:
# ============================================================
# BUILD AND PRINT THE SELECTED NORMAL-ONLY REFERENCE TEXT
#
# Only the frozen NORMAL profile is used.
#
# Included base metrics:
#   - num_offsets
#   - offset_mean
#   - offset_median
#   - offset_max
#   - offset_p75
#   - offset_p90
#   - percent_above_1_5
#   - clean_overlap_seconds
#
# Included global metrics:
#   - best_B_correction_shift_seconds__mean
#   - best_B_correction_shift_seconds__median
#   - estimated_B_lateness_seconds__mean
#   - estimated_B_lateness_seconds__median
#   - alignment_score_gain_vs_zero__mean
#   - alignment_score_gain_vs_zero__median
#   - best_num_bilateral_events__mean
#   - best_event_coverage__mean
#
# No other frozen statistics are exposed.
# ============================================================

import json
import math
import numbers


# ============================================================
# STRICT NORMAL PROFILE AUDIT
# ============================================================

assert isinstance(
    frozen_normal_base_statistics,
    dict,
)


assert isinstance(
    frozen_normal_shift_statistics,
    dict,
)


assert (
    frozen_normal_base_statistics[
        "reference_profile"
    ]
    == "NORMAL"
)


assert (
    frozen_normal_shift_statistics[
        "reference_profile"
    ]
    == "NORMAL"
)


# ============================================================
# NUMERIC VALIDATION HELPER
# ============================================================

def require_finite_numeric(
    record,
    field_name,
):
    """
    Read one required numeric field and verify
    that it exists and contains a finite number.
    """

    assert field_name in record, (
        f"Missing required field: {field_name}"
    )


    value = record[
        field_name
    ]


    assert isinstance(
        value,
        numbers.Real,
    ) and not isinstance(
        value,
        bool,
    ), (
        f"Expected numeric value for {field_name}, "
        f"found {type(value).__name__}: {value}"
    )


    value = float(
        value
    )


    assert math.isfinite(
        value
    ), (
        f"Non-finite value for {field_name}: {value}"
    )


    return value


def normalize_negative_zero(
    value,
):
    """
    Convert -0.0 to 0.0 for cleaner prompt text.
    """

    if abs(
        value
    ) < 1e-12:

        return 0.0


    return value


# ============================================================
# SELECT ONLY THE REQUIRED NORMAL BASE STATISTICS
# ============================================================

normal_num_offsets = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "num_offsets",
    )
)


normal_offset_mean = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_mean",
    )
)


normal_offset_median = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_median",
    )
)


normal_offset_max = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_max",
    )
)


normal_offset_p75 = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_p75",
    )
)


normal_offset_p90 = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_p90",
    )
)


normal_percent_above_1_5 = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "percent_above_1_5",
    )
)


normal_clean_overlap_seconds = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "clean_overlap_seconds",
    )
)


# ============================================================
# SELECT ONLY THE REQUIRED NORMAL GLOBAL-SHIFT STATISTICS
#
# IMPORTANT:
# These fields use a double underscore before
# "mean" and "median".
# ============================================================

normal_best_shift_mean = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "best_B_correction_shift_seconds__mean",
    )
)


normal_best_shift_median = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "best_B_correction_shift_seconds__median",
    )
)


normal_lateness_mean = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "estimated_B_lateness_seconds__mean",
    )
)


normal_lateness_median = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "estimated_B_lateness_seconds__median",
    )
)


normal_alignment_gain_mean = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "alignment_score_gain_vs_zero__mean",
    )
)


normal_alignment_gain_median = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "alignment_score_gain_vs_zero__median",
    )
)


normal_bilateral_events_mean = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "best_num_bilateral_events__mean",
    )
)


normal_event_coverage_fraction = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "best_event_coverage__mean",
    )
)


# ============================================================
# CLEAN DISPLAY VALUES
# ============================================================

normal_best_shift_mean = (
    normalize_negative_zero(
        normal_best_shift_mean
    )
)


normal_best_shift_median = (
    normalize_negative_zero(
        normal_best_shift_median
    )
)


normal_lateness_mean = (
    normalize_negative_zero(
        normal_lateness_mean
    )
)


normal_lateness_median = (
    normalize_negative_zero(
        normal_lateness_median
    )
)


normal_event_coverage_percent = (
    100.0
    * normal_event_coverage_fraction
)


# ============================================================
# STORE THE EXACT VALUES USED IN THE PROMPT
# ============================================================

normal_base_reference_values = {
    "num_signed_offsets": (
        normal_num_offsets
    ),

    "mean_signed_offset_seconds": (
        normal_offset_mean
    ),

    "median_signed_offset_seconds": (
        normal_offset_median
    ),

    "maximum_signed_offset_seconds": (
        normal_offset_max
    ),

    "p75_signed_offset_seconds": (
        normal_offset_p75
    ),

    "p90_signed_offset_seconds": (
        normal_offset_p90
    ),

    "percent_offsets_above_1_5_seconds": (
        normal_percent_above_1_5
    ),

    "filtered_clean_overlap_seconds": (
        normal_clean_overlap_seconds
    ),
}


normal_global_reference_values = {
    "mean_best_B_correction_shift_seconds": (
        normal_best_shift_mean
    ),

    "median_best_B_correction_shift_seconds": (
        normal_best_shift_median
    ),

    "mean_estimated_B_lateness_seconds": (
        normal_lateness_mean
    ),

    "median_estimated_B_lateness_seconds": (
        normal_lateness_median
    ),

    "mean_alignment_score_gain_vs_zero": (
        normal_alignment_gain_mean
    ),

    "median_alignment_score_gain_vs_zero": (
        normal_alignment_gain_median
    ),

    "mean_num_bilateral_alignment_events": (
        normal_bilateral_events_mean
    ),

    "mean_bilateral_event_coverage_percent": (
        normal_event_coverage_percent
    ),
}


assert len(
    normal_base_reference_values
) == 8


assert len(
    normal_global_reference_values
) == 8


# ============================================================
# BUILD NORMAL LOCAL-TEMPORAL REFERENCE TEXT
# ============================================================

normal_base_reference_text = f"""
Frozen NORMAL local temporal reference statistics:

- Number of signed offsets is around {normal_num_offsets:.2f}.
- Mean signed offset is around {normal_offset_mean:.2f} seconds.
- Median signed offset is around {normal_offset_median:.2f} seconds.
- Maximum signed offset is around {normal_offset_max:.2f} seconds.
- P75 signed offset is around {normal_offset_p75:.2f} seconds.
- P90 signed offset is around {normal_offset_p90:.2f} seconds.
- Approximately {normal_percent_above_1_5:.1f}% of offsets are above 1.5 seconds.
- Filtered clean overlap is around {normal_clean_overlap_seconds:.2f} seconds.

These values were calculated only from the frozen NORMAL reference conversations.
They are soft reference patterns and must not be treated as hard classification thresholds.
""".strip()


# ============================================================
# BUILD NORMAL GLOBAL-ALIGNMENT REFERENCE TEXT
# ============================================================

normal_global_reference_text = f"""
Frozen NORMAL global alignment-shift reference statistics:

- Mean best B correction shift is around {normal_best_shift_mean:.2f} seconds.
- Median best B correction shift is around {normal_best_shift_median:.2f} seconds.
- Mean estimated B lateness is around {normal_lateness_mean:.2f} seconds.
- Median estimated B lateness is around {normal_lateness_median:.2f} seconds.
- Mean alignment score gain versus zero shift is around {normal_alignment_gain_mean:.3f}.
- Median alignment score gain versus zero shift is around {normal_alignment_gain_median:.3f}.
- Mean number of bilateral alignment events is around {normal_bilateral_events_mean:.2f}.
- Mean bilateral event coverage is around {normal_event_coverage_percent:.2f}%.

These values were calculated only from the frozen NORMAL reference conversations.
They describe typical global-alignment behavior in the frozen NORMAL set and are not hard classification thresholds.
""".strip()


# ============================================================
# PRINT BOTH PROMPT SECTIONS
# ============================================================

print("=" * 88)
print("normal_base_reference_text")
print("=" * 88)

print(
    normal_base_reference_text
)


print("\n" + "=" * 88)
print("normal_global_reference_text")
print("=" * 88)

print(
    normal_global_reference_text
)


# ============================================================
# PRINT THE EXACT STRUCTURED VALUES USED
# ============================================================

print("\n" + "=" * 88)
print("EXACT SELECTED NORMAL BASE VALUES")
print("=" * 88)

print(
    json.dumps(
        normal_base_reference_values,
        indent=2,
        ensure_ascii=False,
    )
)


print("\n" + "=" * 88)
print("EXACT SELECTED NORMAL GLOBAL VALUES")
print("=" * 88)

print(
    json.dumps(
        normal_global_reference_values,
        indent=2,
        ensure_ascii=False,
    )
)


# ============================================================
# STRICT CONTENT AUDIT
# ============================================================

combined_reference_text = (
    normal_base_reference_text
    + "\n"
    + normal_global_reference_text
)


# No anomaly-specific labels or profiles.
for forbidden_text in [
    "LAG +1",
    "LAG +2",
    "LAG +3",
    "wrong_partner",
    "silent_partner",
    "anomaly_type",
]:

    assert (
        forbidden_text.lower()
        not in combined_reference_text.lower()
    )


# Unwanted frozen reference fields must not appear.
for excluded_text in [
    "original A turns",
    "original B turns",
    "removed A backchannels",
    "removed B backchannels",
    "filtered A turns",
    "filtered B turns",
    "number of negative",
    "number of positive",
    "minimum signed offset",
    "clean overlap percentage",
]:

    assert (
        excluded_text.lower()
        not in combined_reference_text.lower()
    )


# All required selected metrics must appear.
for required_text in [
    "Number of signed offsets",
    "Mean signed offset",
    "Median signed offset",
    "Maximum signed offset",
    "P75 signed offset",
    "P90 signed offset",
    "offsets are above 1.5 seconds",
    "Filtered clean overlap",
    "Mean best B correction shift",
    "Median best B correction shift",
    "Mean estimated B lateness",
    "Median estimated B lateness",
    "Mean alignment score gain",
    "Median alignment score gain",
    "Mean number of bilateral alignment events",
    "Mean bilateral event coverage",
]:

    assert (
        required_text
        in combined_reference_text
    )


print("\n" + "=" * 88)
print("SELECTED NORMAL-ONLY REFERENCE TEXT READY")
print("=" * 88)

print(
    "Base reference metrics included:",
    len(
        normal_base_reference_values
    ),
)

print(
    "Global reference metrics included:",
    len(
        normal_global_reference_values
    ),
)

print(
    "No additional frozen statistics were exposed."
)

print(
    "No explicit anomaly profile or anomaly type was included."
)

normal_base_reference_text
Frozen NORMAL local temporal reference statistics:

- Number of signed offsets is around 5.76.
- Mean signed offset is around 0.30 seconds.
- Median signed offset is around 0.26 seconds.
- Maximum signed offset is around 1.05 seconds.
- P75 signed offset is around 0.59 seconds.
- P90 signed offset is around 0.83 seconds.
- Approximately 6.7% of offsets are above 1.5 seconds.
- Filtered clean overlap is around 6.25 seconds.

These values were calculated only from the frozen NORMAL reference conversations.
They are soft reference patterns and must not be treated as hard classification thresholds.

normal_global_reference_text
Frozen NORMAL global alignment-shift reference statistics:

- Mean best B correction shift is around -0.23 seconds.
- Median best B correction shift is around 0.00 seconds.
- Mean estimated B lateness is around 0.50 seconds.
- Median estimated B lateness is around 0.00 seconds.
- Mean alignment score gain versus zero shift is around 0.023.

# Shared Structured R1 projection and evaluation helpers

The cell below retains the exact database projection, semantic field selection,
Qwen call, structured-output parser, cache format and evaluator used by the prior
Structured R1 experiments.

The improved experiment will apply an additional model-facing projection that removes
filtered turns and overlap while retaining `speaks`, local offsets, all global features,
and the complete coarse + focused semantic summaries.


In [ ]:

# ============================================================
# SHARED STRUCTURED-REASONING EXPERIMENT HELPERS
#
# The complete original Structured R1 projection is retained here
# as an internal source projection. The manual semantic-ablation helper defined later creates a deep-copied
# model-facing view that removes filtered turns, overlap and semantic summaries,
# retains participant-level `speaks`, and preserves local offsets and global features.
#
# The three source prompts are loaded from their exact saved
# prompt_template.txt files.
#
# The only prompt change is replacement of the original binary
# OUTPUT block with one common structured-reasoning JSON schema.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import copy
import hashlib
import json
import re
import time

import pandas as pd
import torch

from tqdm.auto import tqdm


import difflib

MAX_NEW_TOKENS_REASONING = 256
MAX_NEW_TOKENS = MAX_NEW_TOKENS_REASONING

LABELS = [
    "NORMAL",
    "ANOMALOUS",
]


# ============================================================
# EXACT SEMANTIC FIELDS PASSED TO THE MODEL
# ============================================================

COARSE_FIELDS = [
    "speech_content_summary",
    "apparent_topic",
]


FOCUSED_FIELDS = [
    "detailed_speech_summary",
    "main_topic",
    "secondary_topics",
    "key_semantic_details",
    "summary_specificity",
    "unclear_content",
    "confidence",
]


SEGMENT_MAP = {
    "segment_0_0_to_60_seconds": (
        "segment_0"
    ),

    "segment_1_60_to_120_seconds": (
        "segment_1"
    ),
}


# ============================================================
# EXACT TEMPORAL FIELDS PASSED TO THE MODEL
# ============================================================

LOCAL_FEATURE_FIELDS = [
    "clean_overlap_seconds",
    "clean_overlap_percent",

    "signed_strict_offsets_seconds",
    "num_signed_strict_offsets",

    "offset_mean_seconds",
    "offset_median_seconds",
    "offset_max_seconds",
    "offset_p75_seconds",
    "offset_p90_seconds",

    "num_offsets_above_1_5_seconds",
    "percent_offsets_above_1_5_seconds",
]


GLOBAL_FEATURE_FIELDS = [
    "best_B_correction_shift_seconds",
    "estimated_B_lateness_seconds",
    "alignment_score_gain_vs_zero",
    "best_num_bilateral_events",
    "best_event_coverage",
]


# ============================================================
# FIELDS THAT MUST NEVER APPEAR IN THE MODEL PROMPT
# ============================================================

FORBIDDEN_PROMPT_KEYS = [
    "case_id",
    "source_group_id",
    "pair_index",

    "gold_binary_label",
    "gold_anomaly_type",

    "case_variant",
    "pairing_type",

    "conversation_id",
    "participant_id",
    "metadata_path",

    "num_raw_vad_entries",

    "A_source_conversation",
    "B_source_conversation",

    "semantic_summary_source",
    "semantic_summary_coverage",
]


# ============================================================
# GENERAL HELPERS
# ============================================================

def canonical_json(
    value,
):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    )


def sha256_text(
    text,
):
    return hashlib.sha256(
        text.encode(
            "utf-8"
        )
    ).hexdigest()


def select_exact_fields(
    record,
    fields,
    context,
):
    assert isinstance(
        record,
        dict,
    ), (
        f"Expected dictionary for {context}"
    )


    missing_fields = [
        field

        for field in fields

        if field not in record
    ]


    assert not missing_fields, (
        f"Missing fields in {context}: "
        f"{missing_fields}"
    )


    return {
        field: copy.deepcopy(
            record[
                field
            ]
        )

        for field in fields
    }


# ============================================================
# TURN PROJECTION
#
# Database format:
#   {"start": 1.2, "end": 3.4}
#
# Prompt format:
#   [1.2, 3.4]
# ============================================================

def compact_turns(
    turns,
    context,
):
    assert isinstance(
        turns,
        list,
    ), (
        f"Expected list for {context}"
    )


    compact = []


    for index, turn in enumerate(
        turns
    ):

        assert isinstance(
            turn,
            dict,
        ), (
            f"Invalid turn in {context} "
            f"at index {index}"
        )


        assert "start" in turn
        assert "end" in turn


        start = float(
            turn[
                "start"
            ]
        )


        end = float(
            turn[
                "end"
            ]
        )


        assert end >= start


        compact.append([
            start,
            end,
        ])


    return compact


# ============================================================
# SEMANTIC INPUT PROJECTION
#
# Participant IDs and other metadata are deliberately excluded.
# ============================================================

def build_semantic_input(
    case,
):
    semantic_input = {}


    for role in [
        "participant_A",
        "participant_B",
    ]:

        participant_semantics = (
            case[
                "semantic_summaries"
            ][role]
        )


        role_output = {}


        for (
            database_segment_name,
            prompt_segment_name,
        ) in SEGMENT_MAP.items():

            segment_record = (
                participant_semantics[
                    database_segment_name
                ]
            )


            coarse_summary = (
                select_exact_fields(
                    segment_record[
                        "coarse_summary"
                    ],

                    COARSE_FIELDS,

                    (
                        f"{role}/"
                        f"{database_segment_name}/"
                        "coarse_summary"
                    ),
                )
            )


            focused_summary = (
                select_exact_fields(
                    segment_record[
                        "focused_summary"
                    ],

                    FOCUSED_FIELDS,

                    (
                        f"{role}/"
                        f"{database_segment_name}/"
                        "focused_summary"
                    ),
                )
            )


            # Legacy focused-summary speaks must not exist.
            assert (
                "speaks"
                not in focused_summary
            )


            role_output[
                prompt_segment_name
            ] = {
                "coarse_summary": (
                    coarse_summary
                ),

                "focused_summary": (
                    focused_summary
                ),
            }


        semantic_input[
            role
        ] = role_output


    return semantic_input


# ============================================================
# BUILD THE EXACT MODEL INPUT
# ============================================================

def build_binary_model_input(
    case,
):
    local_features = (
        select_exact_fields(
            case[
                "local_temporal_features"
            ],

            LOCAL_FEATURE_FIELDS,

            "local_temporal_features",
        )
    )


    global_features_raw = (
        select_exact_fields(
            case[
                "global_shift_features"
            ],

            GLOBAL_FEATURE_FIELDS,

            "global_shift_features",
        )
    )


    # Stored in the database as a fraction.
    # Presented in the prompt as a percentage so that it is
    # directly comparable with the frozen NORMAL percentage.
    event_coverage_fraction = float(
        global_features_raw.pop(
            "best_event_coverage"
        )
    )


    global_features = {
        **global_features_raw,

        "best_event_coverage_percent": round(
            100.0
            * event_coverage_fraction,

            6,
        ),
    }


    payload = {
        "analysis_duration_seconds": float(
            case[
                "analysis_duration_seconds"
            ]
        ),

        "participant_A": {
            "speaks": bool(
                case[
                    "participant_A"
                ][
                    "speaks"
                ]
            ),

            "filtered_turns": compact_turns(
                case[
                    "participant_A_filtered_turns"
                ],

                "participant_A_filtered_turns",
            ),
        },

        "participant_B": {
            "speaks": bool(
                case[
                    "participant_B"
                ][
                    "speaks"
                ]
            ),

            "filtered_turns": compact_turns(
                case[
                    "participant_B_filtered_turns"
                ],

                "participant_B_filtered_turns",
            ),
        },

        "local_temporal_features": (
            local_features
        ),

        "global_shift_features": (
            global_features
        ),

        "semantic_summaries": (
            build_semantic_input(
                case
            )
        ),
    }


    assert set(
        payload
    ) == {
        "analysis_duration_seconds",
        "participant_A",
        "participant_B",
        "local_temporal_features",
        "global_shift_features",
        "semantic_summaries",
    }


    return payload


# ============================================================
# BUILD THE FINAL PROMPT FOR ONE CASE
# ============================================================

def build_binary_prompt_from_template(
    case,
    prompt_template,
):
    payload = build_binary_model_input(
        case
    )


    prompt = (
        prompt_template
        .format(
            normal_base_reference_text=(
                normal_base_reference_text
            ),

            normal_global_reference_text=(
                normal_global_reference_text
            ),

            duration_seconds=(
                payload[
                    "analysis_duration_seconds"
                ]
            ),

            participant_A_speaks=(
                canonical_json(
                    payload[
                        "participant_A"
                    ][
                        "speaks"
                    ]
                )
            ),

            participant_A_turns=(
                canonical_json(
                    payload[
                        "participant_A"
                    ][
                        "filtered_turns"
                    ]
                )
            ),

            participant_B_speaks=(
                canonical_json(
                    payload[
                        "participant_B"
                    ][
                        "speaks"
                    ]
                )
            ),

            participant_B_turns=(
                canonical_json(
                    payload[
                        "participant_B"
                    ][
                        "filtered_turns"
                    ]
                )
            ),

            local_temporal_features=(
                json.dumps(
                    payload[
                        "local_temporal_features"
                    ],
                    indent=2,
                    ensure_ascii=False,
                )
            ),

            global_shift_features=(
                json.dumps(
                    payload[
                        "global_shift_features"
                    ],
                    indent=2,
                    ensure_ascii=False,
                )
            ),

            semantic_summaries=(
                json.dumps(
                    payload[
                        "semantic_summaries"
                    ],
                    indent=2,
                    ensure_ascii=False,
                )
            ),
        )
    )


    # Strict leakage audit.
    prompt_lower = prompt.lower()


    for forbidden_key in (
        FORBIDDEN_PROMPT_KEYS
    ):

        assert (
            forbidden_key.lower()
            not in prompt_lower
        ), (
            "Forbidden field name leaked into prompt: "
            f"{forbidden_key}"
        )


    return (
        prompt,
        payload,
    )


# ============================================================
# TEXT-ONLY QWEN INFERENCE
#
# The existing qwen_text_only helper used 700 output tokens.
# Here only 64 are needed because the expected output is:
#
#   {"label": "NORMAL"}
#
# or:
#
#   {"label": "ANOMALOUS"}
# ============================================================

def qwen_text_only_binary(
    prompt,
    max_new_tokens=MAX_NEW_TOKENS,
):
    messages = [
        {
            "role": "user",

            "content": [
                {
                    "type": "text",
                    "text": prompt,
                },
            ],
        }
    ]


    inputs = (
        processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",

            processor_kwargs={
                "padding": True,
            },
        )
    )


    inputs = {
        key: (
            value.to(
                model.device
            )
            if hasattr(
                value,
                "to",
            )
            else value
        )

        for key, value
        in inputs.items()
    }


    input_token_count = int(
        inputs[
            "input_ids"
        ].shape[-1]
    )


    with torch.inference_mode():

        output_ids = model.generate(
            **inputs,

            max_new_tokens=(
                max_new_tokens
            ),

            do_sample=False,

            use_cache=True,
        )


    generated_ids = output_ids[
        :,
        inputs[
            "input_ids"
        ].shape[1]:,
    ]


    raw_output = (
        processor.batch_decode(
            generated_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )[0]
    )


    return (
        raw_output,
        input_token_count,
    )


# ============================================================
# ROBUST JSON PARSING
# ============================================================

def extract_first_json_object(
    text,
):
    cleaned = text.strip()


    cleaned = re.sub(
        r"^```(?:json)?\s*",
        "",
        cleaned,
        flags=re.IGNORECASE,
    )


    cleaned = re.sub(
        r"\s*```$",
        "",
        cleaned,
    )


    try:

        return json.loads(
            cleaned
        )

    except Exception:

        pass


    decoder = json.JSONDecoder()


    possible_starts = [
        index

        for index, character
        in enumerate(
            cleaned
        )

        if character == "{"
    ]


    for start in possible_starts:

        try:

            parsed, _ = (
                decoder.raw_decode(
                    cleaned[
                        start:
                    ]
                )
            )


            return parsed

        except Exception:

            continue


    return None

# ============================================================
# STRUCTURED REASONING SCHEMA
# ============================================================

REASONING_SCHEMA_KEYS = [
    "participation_assessment",
    "local_temporal_assessment",
    "global_temporal_assessment",
    "temporal_assessment",
    "semantic_assessment",
    "decisive_dimension",
    "label",
]


REASONING_ALLOWED_VALUES = {
    "participation_assessment": {
        "VALID",
        "INVALID",
    },

    "local_temporal_assessment": {
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    },

    "global_temporal_assessment": {
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    },

    "temporal_assessment": {
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    },

    "semantic_assessment": {
        "COMPATIBLE",
        "INCOMPATIBLE",
        "LIMITED",
    },

    "decisive_dimension": {
        "PARTICIPATION",
        "TEMPORAL",
        "SEMANTIC",
        "NONE",
    },

    "label": {
        "NORMAL",
        "ANOMALOUS",
    },
}


OLD_BINARY_OUTPUT_BLOCK = """
============================================================
OUTPUT
============================================================

Return ONLY one valid JSON object with exactly this schema:

{{
  "label": "NORMAL or ANOMALOUS"
}}

Do not include confidence.
Do not include reasoning.
Do not include an anomaly type.
Do not include Markdown.
Do not include any text outside the JSON object.
""".strip()


STRUCTURED_REASONING_OUTPUT_BLOCK = """
============================================================
STRUCTURED REASONING OUTPUT
============================================================

Preserve all evidence definitions, comparison rules, decision criteria,
and the final binary decision policy stated above.

Before selecting the final label, expose the following fixed structured
assessments.

These fields are diagnostic outputs only.

They must not introduce new evidence, new thresholds, a new voting rule,
or any change to the existing decision policy.

Use the available evidence exactly as instructed above.

Assessment meanings:

- participation_assessment:
  - VALID
  - INVALID

- local_temporal_assessment:
  - NORMAL
  - ANOMALOUS
  - LIMITED

- global_temporal_assessment:
  - NORMAL
  - ANOMALOUS
  - LIMITED

- temporal_assessment:
  - NORMAL
  - ANOMALOUS
  - LIMITED

- semantic_assessment:
  - COMPATIBLE
  - INCOMPATIBLE
  - LIMITED

- decisive_dimension:
  - PARTICIPATION
  - TEMPORAL
  - SEMANTIC
  - NONE

The local and global temporal assessments must reflect the reliability
rules already defined in the prompt.

The combined temporal assessment must be based on the existing temporal
decision policy.

The final label must follow the existing final decision policy exactly.

Do not infer or output an anomaly subtype.
Do not infer or output a delay magnitude.
Do not provide free-form reasoning.

============================================================
OUTPUT
============================================================

Return ONLY one valid JSON object with exactly this schema:

{{
  "participation_assessment": "VALID or INVALID",
  "local_temporal_assessment": "NORMAL or ANOMALOUS or LIMITED",
  "global_temporal_assessment": "NORMAL or ANOMALOUS or LIMITED",
  "temporal_assessment": "NORMAL or ANOMALOUS or LIMITED",
  "semantic_assessment": "COMPATIBLE or INCOMPATIBLE or LIMITED",
  "decisive_dimension": "PARTICIPATION or TEMPORAL or SEMANTIC or NONE",
  "label": "NORMAL or ANOMALOUS"
}}

Do not include confidence.
Do not include an anomaly type.
Do not include Markdown.
Do not include any text outside the JSON object.
""".strip()


def build_structured_reasoning_prompt_template(
    source_prompt_template,
):
    assert isinstance(
        source_prompt_template,
        str,
    )


    assert source_prompt_template.endswith(
        OLD_BINARY_OUTPUT_BLOCK
    ), (
        "The saved source prompt does not end with the exact "
        "original binary OUTPUT block."
    )


    reasoning_prompt_template = (
        source_prompt_template[
            :-len(
                OLD_BINARY_OUTPUT_BLOCK
            )
        ]
        + STRUCTURED_REASONING_OUTPUT_BLOCK
    )


    reconstructed_source_prompt = (
        reasoning_prompt_template[
            :-len(
                STRUCTURED_REASONING_OUTPUT_BLOCK
            )
        ]
        + OLD_BINARY_OUTPUT_BLOCK
    )


    assert (
        reconstructed_source_prompt
        == source_prompt_template
    ), (
        "Unexpected source-prompt modification."
    )


    return reasoning_prompt_template


def parse_structured_reasoning_prediction(
    raw_output,
):
    parsed = extract_first_json_object(
        raw_output
    )


    normalized = {
        key: None
        for key in REASONING_SCHEMA_KEYS
    }


    schema_errors = []


    if isinstance(
        parsed,
        dict,
    ):

        for key in REASONING_SCHEMA_KEYS:

            if key in parsed:

                normalized[
                    key
                ] = str(
                    parsed[
                        key
                    ]
                ).strip().upper()


        for key in REASONING_SCHEMA_KEYS:

            if normalized[
                key
            ] not in REASONING_ALLOWED_VALUES[
                key
            ]:

                schema_errors.append(
                    (
                        f"{key}: "
                        f"{normalized[key]}"
                    )
                )


        exact_keys = (
            set(
                parsed.keys()
            )
            == set(
                REASONING_SCHEMA_KEYS
            )
        )


        schema_exact = (
            exact_keys
            and
            not schema_errors
        )


        label = normalized[
            "label"
        ]


        if label in LABELS:

            return {
                "prediction": label,

                "parse_mode": (
                    "structured_json"
                ),

                "schema_exact": bool(
                    schema_exact
                ),

                "parsed_output": parsed,

                "schema_errors": (
                    schema_errors
                ),

                **{
                    key: normalized[
                        key
                    ]

                    for key in (
                        REASONING_SCHEMA_KEYS
                    )

                    if key != "label"
                },
            }


    plain_output = (
        raw_output
        .strip()
        .strip('"')
        .strip("'")
        .upper()
    )


    if plain_output in LABELS:

        return {
            "prediction": plain_output,

            "parse_mode": (
                "exact_plaintext_fallback"
            ),

            "schema_exact": False,

            "parsed_output": None,

            "schema_errors": [
                "Structured reasoning fields missing."
            ],

            **{
                key: None

                for key in (
                    REASONING_SCHEMA_KEYS
                )

                if key != "label"
            },
        }


    return {
        "prediction": None,

        "parse_mode": "invalid",

        "schema_exact": False,

        "parsed_output": parsed,

        "schema_errors": (
            schema_errors
            if schema_errors
            else [
                "No valid structured JSON prediction."
            ]
        ),

        **{
            key: normalized[
                key
            ]

            for key in (
                REASONING_SCHEMA_KEYS
            )

            if key != "label"
        },
    }


# ============================================================
# FULL-SEMANTICS PAYLOAD AUDIT
# ============================================================

def collect_nested_keys_reasoning(
    value,
):
    keys = set()


    if isinstance(
        value,
        dict,
    ):

        for key, nested_value in value.items():

            keys.add(
                str(key)
            )


            keys.update(
                collect_nested_keys_reasoning(
                    nested_value
                )
            )


    elif isinstance(
        value,
        list,
    ):

        for item in value:

            keys.update(
                collect_nested_keys_reasoning(
                    item
                )
            )


    return keys


FULL_SEMANTIC_REQUIRED_KEYS = [
    "semantic_summaries",
    "coarse_summary",
    "focused_summary",

    "speech_content_summary",
    "apparent_topic",

    "detailed_speech_summary",
    "main_topic",
    "secondary_topics",
    "key_semantic_details",
    "summary_specificity",
    "unclear_content",
    "confidence",
]


# ============================================================
# EXACT PROMPT CONFIGURATION
# ============================================================

def prepare_reasoning_experiment(
    *,
    experiment_version,
    source_experiment_name,
    experiment_title,
    required_source_markers,
    forbidden_source_markers,
    temporal_profiles_used,
    assessment_policy,
):
    source_experiment_dir = (
        OUT_DIR
        / source_experiment_name
    )


    source_prompt_path = (
        source_experiment_dir
        / "prompt_template.txt"
    )


    assert source_prompt_path.exists(), (
        "Saved source prompt not found:\n"
        f"{source_prompt_path}"
    )


    source_prompt_template = (
        source_prompt_path.read_text(
            encoding="utf-8"
        )
    )


    for marker in required_source_markers:

        assert marker in source_prompt_template, (
            "Required source-prompt marker missing: "
            f"{marker}"
        )


    for marker in forbidden_source_markers:

        assert marker not in source_prompt_template, (
            "Forbidden source-prompt marker found: "
            f"{marker}"
        )


    for semantic_marker in [
        "Coarse semantic information:",
        "Focused semantic information:",
        "- detailed_speech_summary",
        "- main_topic",
        "- secondary_topics",
        "- key_semantic_details",
    ]:

        assert semantic_marker in source_prompt_template, (
            "The source prompt is not a full-semantics prompt. "
            f"Missing: {semantic_marker}"
        )


    reasoning_prompt_template = (
        build_structured_reasoning_prompt_template(
            source_prompt_template
        )
    )


    experiment_dir = (
        OUT_DIR
        / experiment_version
    )


    experiment_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    paths = {
        "prediction_cache": (
            experiment_dir
            / "predictions_cache.json"
        ),

        "predictions_csv": (
            experiment_dir
            / "predictions_all_400.csv"
        ),

        "metrics_json": (
            experiment_dir
            / "metrics.json"
        ),

        "confusion_matrix_csv": (
            experiment_dir
            / "confusion_matrix.csv"
        ),

        "family_metrics_csv": (
            experiment_dir
            / "metrics_by_case_family.csv"
        ),

        "variant_metrics_csv": (
            experiment_dir
            / "metrics_by_case_variant.csv"
        ),

        "errors_csv": (
            experiment_dir
            / "classification_errors.csv"
        ),

        "reasoning_assessments_csv": (
            experiment_dir
            / "reasoning_assessments.csv"
        ),

        "reasoning_inconsistencies_csv": (
            experiment_dir
            / "reasoning_inconsistencies.csv"
        ),

        "assessment_distributions_csv": (
            experiment_dir
            / "assessment_distributions.csv"
        ),

        "prompt_template": (
            experiment_dir
            / "prompt_template.txt"
        ),

        "source_prompt_copy": (
            experiment_dir
            / "source_prompt_template.txt"
        ),

        "prompt_diff": (
            experiment_dir
            / "prompt_diff_vs_source.txt"
        ),

        "experiment_manifest": (
            experiment_dir
            / "experiment_manifest.json"
        ),
    }


    source_prompt_sha256 = sha256_text(
        source_prompt_template
    )


    reasoning_prompt_sha256 = sha256_text(
        reasoning_prompt_template
    )


    reasoning_schema_sha256 = sha256_text(
        STRUCTURED_REASONING_OUTPUT_BLOCK
    )


    normal_reference_sha256 = sha256_text(
        normal_base_reference_text
        + "\n"
        + normal_global_reference_text
    )


    paths[
        "prompt_template"
    ].write_text(
        reasoning_prompt_template,
        encoding="utf-8",
    )


    paths[
        "source_prompt_copy"
    ].write_text(
        source_prompt_template,
        encoding="utf-8",
    )


    prompt_diff_lines = difflib.unified_diff(
        source_prompt_template.splitlines(),
        reasoning_prompt_template.splitlines(),

        fromfile=(
            f"{source_experiment_name}/prompt_template.txt"
        ),

        tofile=(
            f"{experiment_version}/prompt_template.txt"
        ),

        lineterm="",
    )


    paths[
        "prompt_diff"
    ].write_text(
        "\n".join(
            prompt_diff_lines
        ),
        encoding="utf-8",
    )


    example_prompt, example_payload = (
        build_binary_prompt_from_template(
            consolidation_cases[0],
            reasoning_prompt_template,
        )
    )


    payload_keys = collect_nested_keys_reasoning(
        example_payload
    )


    for required_key in FULL_SEMANTIC_REQUIRED_KEYS:

        assert required_key in payload_keys, (
            "Full-semantic input field missing: "
            f"{required_key}"
        )


    assert (
        "STRUCTURED REASONING OUTPUT"
        in example_prompt
    )


    assert (
        "Do not include reasoning."
        not in reasoning_prompt_template[
            -len(
                STRUCTURED_REASONING_OUTPUT_BLOCK
            ):
        ]
    )


    manifest = {
        "experiment_version": (
            experiment_version
        ),

        "experiment_title": (
            experiment_title
        ),

        "source_experiment": (
            source_experiment_name
        ),

        "source_prompt_path": str(
            source_prompt_path
        ),

        "source_prompt_sha256": (
            source_prompt_sha256
        ),

        "reasoning_prompt_sha256": (
            reasoning_prompt_sha256
        ),

        "reasoning_schema_sha256": (
            reasoning_schema_sha256
        ),

        "normal_reference_sha256": (
            normal_reference_sha256
        ),

        "semantic_input": (
            "coarse_and_focused"
        ),

        "focused_summaries_used": True,

        "temporal_profiles_used": (
            temporal_profiles_used
        ),

        "assessment_policy": (
            assessment_policy
        ),

        "max_new_tokens": (
            MAX_NEW_TOKENS_REASONING
        ),

        "model_id": (
            MODEL_ID
        ),

        "decoding": {
            "do_sample": False,
            "use_cache": True,
        },

        "only_prompt_change": (
            "The exact original binary OUTPUT block was "
            "replaced by the common structured-reasoning "
            "OUTPUT block."
        ),

        "all_model_facing_evidence_unchanged": (
            True
        ),

        "all_pre_output_prompt_text_unchanged": (
            True
        ),
    }


    paths[
        "experiment_manifest"
    ].write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


    config = {
        **manifest,

        "experiment_dir": (
            experiment_dir
        ),

        "source_prompt_template": (
            source_prompt_template
        ),

        "reasoning_prompt_template": (
            reasoning_prompt_template
        ),

        "paths": paths,
    }


    print("=" * 88)
    print(
        f"{experiment_title} — CONFIGURATION READY"
    )
    print("=" * 88)

    print(
        "Experiment version:",
        experiment_version,
    )

    print(
        "Source experiment:",
        source_experiment_name,
    )

    print(
        "Source prompt SHA256:",
        source_prompt_sha256,
    )

    print(
        "Reasoning prompt SHA256:",
        reasoning_prompt_sha256,
    )

    print(
        "Semantic input:",
        "coarse_and_focused",
    )

    print(
        "Focused summaries used:",
        True,
    )

    print(
        "Temporal profiles:",
        temporal_profiles_used,
    )

    print(
        "Assessment policy:",
        assessment_policy,
    )

    print(
        "Only source-prompt change:",
        (
            "binary OUTPUT block -> "
            "structured reasoning OUTPUT block"
        ),
    )

    print(
        "Example prompt characters:",
        len(
            example_prompt
        ),
    )

    print(
        "Prediction cache:",
        paths[
            "prediction_cache"
        ],
    )

    print(
        "Existing cache:",
        paths[
            "prediction_cache"
        ].exists(),
    )


    return config


# ============================================================
# CHECKPOINTED INFERENCE
# ============================================================

def reasoning_utc_now():
    return datetime.now(
        timezone.utc
    ).isoformat()


def reasoning_atomic_write_json(
    path,
    data,
):
    path = Path(
        path
    )


    temporary_path = path.with_suffix(
        path.suffix
        + ".tmp"
    )


    temporary_path.write_text(
        json.dumps(
            data,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


    temporary_path.replace(
        path
    )


def create_reasoning_cache(
    config,
):
    return {
        "experiment_version": (
            config[
                "experiment_version"
            ]
        ),

        "experiment_title": (
            config[
                "experiment_title"
            ]
        ),

        "source_experiment": (
            config[
                "source_experiment"
            ]
        ),

        "model_id": (
            MODEL_ID
        ),

        "source_prompt_sha256": (
            config[
                "source_prompt_sha256"
            ]
        ),

        "reasoning_prompt_sha256": (
            config[
                "reasoning_prompt_sha256"
            ]
        ),

        "reasoning_schema_sha256": (
            config[
                "reasoning_schema_sha256"
            ]
        ),

        "normal_reference_sha256": (
            config[
                "normal_reference_sha256"
            ]
        ),

        "semantic_input": (
            "coarse_and_focused"
        ),

        "focused_summaries_used": True,

        "temporal_profiles_used": (
            config[
                "temporal_profiles_used"
            ]
        ),

        "assessment_policy": (
            config[
                "assessment_policy"
            ]
        ),

        "max_new_tokens": (
            MAX_NEW_TOKENS_REASONING
        ),

        "created_at_utc": (
            reasoning_utc_now()
        ),

        "updated_at_utc": (
            reasoning_utc_now()
        ),

        "records": {},
    }


def run_reasoning_experiment(
    config,
):
    prediction_cache_path = (
        config[
            "paths"
        ][
            "prediction_cache"
        ]
    )


    if prediction_cache_path.exists():

        prediction_cache = json.loads(
            prediction_cache_path.read_text(
                encoding="utf-8"
            )
        )


        for key in [
            "experiment_version",
            "source_experiment",
            "model_id",
            "source_prompt_sha256",
            "reasoning_prompt_sha256",
            "reasoning_schema_sha256",
            "normal_reference_sha256",
            "semantic_input",
            "focused_summaries_used",
            "temporal_profiles_used",
            "assessment_policy",
            "max_new_tokens",
        ]:

            expected_value = (
                create_reasoning_cache(
                    config
                )[
                    key
                ]
            )


            assert (
                prediction_cache[
                    key
                ]
                == expected_value
            ), (
                f"Cache mismatch for {key}"
            )


        print(
            "Resuming cache:",
            prediction_cache_path,
        )

        print(
            "Existing records:",
            len(
                prediction_cache[
                    "records"
                ]
            ),
        )


    else:

        prediction_cache = (
            create_reasoning_cache(
                config
            )
        )


        reasoning_atomic_write_json(
            prediction_cache_path,
            prediction_cache,
        )


        print(
            "Created cache:",
            prediction_cache_path,
        )


    ordered_cases = sorted(
        consolidation_cases,

        key=lambda case: str(
            case[
                "case_id"
            ]
        ),
    )


    assert len(
        ordered_cases
    ) == 400


    for case in tqdm(
        ordered_cases,
        desc=(
            config[
                "experiment_version"
            ]
        ),
    ):

        case_id = str(
            case[
                "case_id"
            ]
        )


        prompt, model_input_payload = (
            build_binary_prompt_from_template(
                case,
                config[
                    "reasoning_prompt_template"
                ],
            )
        )


        current_payload_keys = (
            collect_nested_keys_reasoning(
                model_input_payload
            )
        )


        for required_key in (
            FULL_SEMANTIC_REQUIRED_KEYS
        ):

            assert required_key in (
                current_payload_keys
            ), (
                f"Missing full-semantic field "
                f"for case {case_id}: "
                f"{required_key}"
            )


        prompt_sha256 = sha256_text(
            prompt
        )


        input_payload_sha256 = sha256_text(
            canonical_json(
                model_input_payload
            )
        )


        existing_record = (
            prediction_cache[
                "records"
            ].get(
                case_id
            )
        )


        if (
            existing_record is not None
            and
            existing_record.get(
                "prediction"
            ) in LABELS
        ):

            assert (
                existing_record[
                    "prompt_sha256"
                ]
                == prompt_sha256
            )


            assert (
                existing_record[
                    "input_payload_sha256"
                ]
                == input_payload_sha256
            )


            continue


        started = time.perf_counter()


        try:

            raw_output, input_token_count = (
                qwen_text_only_binary(
                    prompt,
                    max_new_tokens=(
                        MAX_NEW_TOKENS_REASONING
                    ),
                )
            )


            parsed_result = (
                parse_structured_reasoning_prediction(
                    raw_output
                )
            )


            generation_error = None


        except Exception as exc:

            raw_output = ""


            input_token_count = None


            parsed_result = {
                "prediction": None,
                "parse_mode": "generation_error",
                "schema_exact": False,
                "parsed_output": None,
                "schema_errors": [],
                "participation_assessment": None,
                "local_temporal_assessment": None,
                "global_temporal_assessment": None,
                "temporal_assessment": None,
                "semantic_assessment": None,
                "decisive_dimension": None,
            }


            generation_error = (
                f"{type(exc).__name__}: {exc}"
            )


            if torch.cuda.is_available():

                torch.cuda.empty_cache()


        elapsed_seconds = (
            time.perf_counter()
            - started
        )


        prediction_cache[
            "records"
        ][case_id] = {
            "case_id": (
                case_id
            ),

            "source_group_id": str(
                case[
                    "source_group_id"
                ]
            ),

            "case_family": (
                get_case_family(
                    case
                )
            ),

            "case_variant": str(
                case[
                    "case_variant"
                ]
            ),

            "gold_binary_label": str(
                case[
                    "gold_binary_label"
                ]
            ).upper(),

            "prompt_sha256": (
                prompt_sha256
            ),

            "input_payload_sha256": (
                input_payload_sha256
            ),

            "semantic_input": (
                "coarse_and_focused"
            ),

            "focused_summaries_used": True,

            "input_token_count": (
                input_token_count
            ),

            "max_new_tokens": (
                MAX_NEW_TOKENS_REASONING
            ),

            "raw_output": (
                raw_output
            ),

            "prediction": (
                parsed_result[
                    "prediction"
                ]
            ),

            "participation_assessment": (
                parsed_result[
                    "participation_assessment"
                ]
            ),

            "local_temporal_assessment": (
                parsed_result[
                    "local_temporal_assessment"
                ]
            ),

            "global_temporal_assessment": (
                parsed_result[
                    "global_temporal_assessment"
                ]
            ),

            "temporal_assessment": (
                parsed_result[
                    "temporal_assessment"
                ]
            ),

            "semantic_assessment": (
                parsed_result[
                    "semantic_assessment"
                ]
            ),

            "decisive_dimension": (
                parsed_result[
                    "decisive_dimension"
                ]
            ),

            "parse_mode": (
                parsed_result[
                    "parse_mode"
                ]
            ),

            "schema_exact": bool(
                parsed_result[
                    "schema_exact"
                ]
            ),

            "schema_errors": (
                parsed_result[
                    "schema_errors"
                ]
            ),

            "parsed_output": (
                parsed_result[
                    "parsed_output"
                ]
            ),

            "generation_error": (
                generation_error
            ),

            "elapsed_seconds": round(
                elapsed_seconds,
                4,
            ),

            "completed_at_utc": (
                reasoning_utc_now()
            ),
        }


        prediction_cache[
            "updated_at_utc"
        ] = reasoning_utc_now()


        reasoning_atomic_write_json(
            prediction_cache_path,
            prediction_cache,
        )


    print("\n" + "=" * 88)
    print(
        f"{config['experiment_title']} — INFERENCE COMPLETE"
    )
    print("=" * 88)

    print(
        "Cached records:",
        len(
            prediction_cache[
                "records"
            ]
        ),
    )

    print(
        "Prediction cache:",
        prediction_cache_path,
    )


    return prediction_cache


# ============================================================
# EVALUATION
# ============================================================

def summarize_reasoning_group(
    group,
):
    valid_group = group[
        group[
            "valid_prediction"
        ]
    ]


    return {
        "total_cases": int(
            len(
                group
            )
        ),

        "valid_predictions": int(
            len(
                valid_group
            )
        ),

        "invalid_predictions": int(
            len(
                group
            )
            -
            len(
                valid_group
            )
        ),

        "predicted_NORMAL": int(
            (
                valid_group[
                    "prediction"
                ]
                == "NORMAL"
            ).sum()
        ),

        "predicted_ANOMALOUS": int(
            (
                valid_group[
                    "prediction"
                ]
                == "ANOMALOUS"
            ).sum()
        ),

        "correct_predictions": int(
            group[
                "correct"
            ].sum()
        ),

        "accuracy_on_valid": (
            float(
                (
                    valid_group[
                        "gold_label"
                    ]
                    ==
                    valid_group[
                        "prediction"
                    ]
                ).mean()
            )
            if len(
                valid_group
            )
            else float(
                "nan"
            )
        ),

        "strict_accuracy_invalid_as_wrong": float(
            group[
                "correct"
            ].mean()
        ),

        "exact_schema_rate": float(
            group[
                "schema_exact"
            ].mean()
        ),
    }


def evaluate_reasoning_experiment(
    config,
):
    from sklearn.metrics import (
        accuracy_score,
        balanced_accuracy_score,
        classification_report,
        confusion_matrix,
        f1_score,
        matthews_corrcoef,
        precision_score,
        recall_score,
    )

    from IPython.display import display


    prediction_cache = json.loads(
        config[
            "paths"
        ][
            "prediction_cache"
        ].read_text(
            encoding="utf-8"
        )
    )


    assert (
        prediction_cache[
            "experiment_version"
        ]
        == config[
            "experiment_version"
        ]
    )


    assert (
        prediction_cache[
            "reasoning_prompt_sha256"
        ]
        == config[
            "reasoning_prompt_sha256"
        ]
    )


    result_rows = []


    for case in consolidation_cases:

        case_id = str(
            case[
                "case_id"
            ]
        )


        record = (
            prediction_cache[
                "records"
            ].get(
                case_id
            )
        )


        result_rows.append({
            "case_id": (
                case_id
            ),

            "source_group_id": str(
                case[
                    "source_group_id"
                ]
            ),

            "case_family": (
                get_case_family(
                    case
                )
            ),

            "case_variant": str(
                case[
                    "case_variant"
                ]
            ),

            "gold_label": str(
                case[
                    "gold_binary_label"
                ]
            ).upper(),

            "prediction": (
                None
                if record is None
                else record.get(
                    "prediction"
                )
            ),

            "participation_assessment": (
                None
                if record is None
                else record.get(
                    "participation_assessment"
                )
            ),

            "local_temporal_assessment": (
                None
                if record is None
                else record.get(
                    "local_temporal_assessment"
                )
            ),

            "global_temporal_assessment": (
                None
                if record is None
                else record.get(
                    "global_temporal_assessment"
                )
            ),

            "temporal_assessment": (
                None
                if record is None
                else record.get(
                    "temporal_assessment"
                )
            ),

            "semantic_assessment": (
                None
                if record is None
                else record.get(
                    "semantic_assessment"
                )
            ),

            "decisive_dimension": (
                None
                if record is None
                else record.get(
                    "decisive_dimension"
                )
            ),

            "parse_mode": (
                "missing"
                if record is None
                else record.get(
                    "parse_mode"
                )
            ),

            "schema_exact": (
                False
                if record is None
                else bool(
                    record.get(
                        "schema_exact",
                        False,
                    )
                )
            ),

            "schema_errors": (
                None
                if record is None
                else json.dumps(
                    record.get(
                        "schema_errors",
                        [],
                    ),
                    ensure_ascii=False,
                )
            ),

            "input_token_count": (
                None
                if record is None
                else record.get(
                    "input_token_count"
                )
            ),

            "elapsed_seconds": (
                None
                if record is None
                else record.get(
                    "elapsed_seconds"
                )
            ),

            "raw_output": (
                ""
                if record is None
                else record.get(
                    "raw_output",
                    "",
                )
            ),

            "generation_error": (
                None
                if record is None
                else record.get(
                    "generation_error"
                )
            ),
        })


    results_df = pd.DataFrame(
        result_rows
    )


    results_df[
        "valid_prediction"
    ] = results_df[
        "prediction"
    ].isin(
        LABELS
    )


    results_df[
        "correct"
    ] = (
        results_df[
            "valid_prediction"
        ]
        &
        (
            results_df[
                "gold_label"
            ]
            ==
            results_df[
                "prediction"
            ]
        )
    )


    results_df[
        "normal_with_invalid_participation"
    ] = (
        (
            results_df[
                "prediction"
            ]
            == "NORMAL"
        )
        &
        (
            results_df[
                "participation_assessment"
            ]
            == "INVALID"
        )
    )


    results_df[
        "normal_with_anomalous_temporal"
    ] = (
        (
            results_df[
                "prediction"
            ]
            == "NORMAL"
        )
        &
        (
            results_df[
                "temporal_assessment"
            ]
            == "ANOMALOUS"
        )
    )


    results_df[
        "normal_with_incompatible_semantics"
    ] = (
        (
            results_df[
                "prediction"
            ]
            == "NORMAL"
        )
        &
        (
            results_df[
                "semantic_assessment"
            ]
            == "INCOMPATIBLE"
        )
    )


    results_df[
        "reasoning_inconsistency"
    ] = (
        results_df[
            [
                "normal_with_invalid_participation",
                "normal_with_anomalous_temporal",
                "normal_with_incompatible_semantics",
            ]
        ].any(
            axis=1
        )
    )


    assert len(
        results_df
    ) == 400


    assert (
        results_df[
            "case_id"
        ].is_unique
    )


    valid_df = results_df[
        results_df[
            "valid_prediction"
        ]
    ].copy()


    invalid_df = results_df[
        ~results_df[
            "valid_prediction"
        ]
    ].copy()


    assert len(
        valid_df
    ) > 0


    y_true = valid_df[
        "gold_label"
    ]


    y_pred = valid_df[
        "prediction"
    ]


    confusion = confusion_matrix(
        y_true,
        y_pred,
        labels=[
            "NORMAL",
            "ANOMALOUS",
        ],
    )


    confusion_df = pd.DataFrame(
        confusion,

        index=[
            "Gold NORMAL",
            "Gold ANOMALOUS",
        ],

        columns=[
            "Pred NORMAL",
            "Pred ANOMALOUS",
        ],
    )


    true_normal = int(
        confusion[
            0,
            0,
        ]
    )


    false_anomalous = int(
        confusion[
            0,
            1,
        ]
    )


    normal_recall = (
        true_normal
        /
        (
            true_normal
            + false_anomalous
        )
        if (
            true_normal
            + false_anomalous
        )
        else float(
            "nan"
        )
    )


    metrics = {
        "experiment_version": (
            config[
                "experiment_version"
            ]
        ),

        "source_experiment": (
            config[
                "source_experiment"
            ]
        ),

        "semantic_input": (
            "coarse_and_focused"
        ),

        "focused_summaries_used": True,

        "temporal_profiles_used": (
            config[
                "temporal_profiles_used"
            ]
        ),

        "assessment_policy": (
            config[
                "assessment_policy"
            ]
        ),

        "total_cases": int(
            len(
                results_df
            )
        ),

        "valid_predictions": int(
            len(
                valid_df
            )
        ),

        "invalid_predictions": int(
            len(
                invalid_df
            )
        ),

        "exact_json_schema_rate": float(
            results_df[
                "schema_exact"
            ].mean()
        ),

        "reasoning_inconsistency_count": int(
            results_df[
                "reasoning_inconsistency"
            ].sum()
        ),

        "accuracy_valid_predictions": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),

        "strict_accuracy_invalid_as_wrong": float(
            results_df[
                "correct"
            ].mean()
        ),

        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                y_pred,
            )
        ),

        "anomalous_precision": float(
            precision_score(
                y_true,
                y_pred,
                pos_label="ANOMALOUS",
                zero_division=0,
            )
        ),

        "anomalous_recall": float(
            recall_score(
                y_true,
                y_pred,
                pos_label="ANOMALOUS",
                zero_division=0,
            )
        ),

        "anomalous_f1": float(
            f1_score(
                y_true,
                y_pred,
                pos_label="ANOMALOUS",
                zero_division=0,
            )
        ),

        "normal_recall_specificity": float(
            normal_recall
        ),

        "matthews_correlation_coefficient": float(
            matthews_corrcoef(
                y_true,
                y_pred,
            )
        ),

        "confusion_matrix_label_order": [
            "NORMAL",
            "ANOMALOUS",
        ],

        "confusion_matrix": (
            confusion.tolist()
        ),
    }


    family_rows = []


    for (
        family_name,
        family_group,
    ) in results_df.groupby(
        "case_family",
        sort=True,
    ):

        family_rows.append({
            "case_family": (
                family_name
            ),

            **summarize_reasoning_group(
                family_group
            ),
        })


    family_metrics_df = pd.DataFrame(
        family_rows
    )


    variant_rows = []


    for (
        variant_name,
        variant_group,
    ) in results_df.groupby(
        "case_variant",
        sort=True,
    ):

        variant_rows.append({
            "case_variant": (
                variant_name
            ),

            **summarize_reasoning_group(
                variant_group
            ),
        })


    variant_metrics_df = pd.DataFrame(
        variant_rows
    )


    classification_report_df = pd.DataFrame(
        classification_report(
            y_true,
            y_pred,
            labels=[
                "NORMAL",
                "ANOMALOUS",
            ],
            output_dict=True,
            zero_division=0,
        )
    ).T


    error_df = results_df[
        (
            ~results_df[
                "valid_prediction"
            ]
        )
        |
        (
            results_df[
                "gold_label"
            ] != results_df[
                "prediction"
            ]
        )
    ].copy()


    reasoning_inconsistency_df = (
        results_df[
            results_df[
                "reasoning_inconsistency"
            ]
        ].copy()
    )


    assessment_rows = []


    for assessment_field in [
        "participation_assessment",
        "local_temporal_assessment",
        "global_temporal_assessment",
        "temporal_assessment",
        "semantic_assessment",
        "decisive_dimension",
    ]:

        counts = (
            results_df[
                assessment_field
            ]
            .fillna(
                "MISSING"
            )
            .value_counts(
                dropna=False
            )
        )


        for value, count in counts.items():

            assessment_rows.append({
                "assessment_field": (
                    assessment_field
                ),

                "assessment_value": (
                    value
                ),

                "count": int(
                    count
                ),
            })


    assessment_distributions_df = (
        pd.DataFrame(
            assessment_rows
        )
    )


    source_group_rows = []


    for (
        source_group_id,
        source_group,
    ) in results_df.groupby(
        "source_group_id"
    ):

        source_group_rows.append({
            "source_group_id": (
                source_group_id
            ),

            "num_cases": int(
                len(
                    source_group
                )
            ),

            "all_predictions_valid": bool(
                source_group[
                    "valid_prediction"
                ].all()
            ),

            "all_cases_correct": bool(
                source_group[
                    "valid_prediction"
                ].all()
                and
                source_group[
                    "correct"
                ].all()
            ),
        })


    source_group_df = pd.DataFrame(
        source_group_rows
    )


    assert len(
        source_group_df
    ) == 100


    assert set(
        source_group_df[
            "num_cases"
        ]
    ) == {
        4
    }


    metrics[
        "source_group_exact_match_rate"
    ] = float(
        source_group_df[
            "all_cases_correct"
        ].mean()
    )


    paths = config[
        "paths"
    ]


    results_df.to_csv(
        paths[
            "predictions_csv"
        ],
        index=False,
    )


    reasoning_columns = [
        "case_id",
        "source_group_id",
        "case_family",
        "case_variant",
        "gold_label",
        "prediction",
        "participation_assessment",
        "local_temporal_assessment",
        "global_temporal_assessment",
        "temporal_assessment",
        "semantic_assessment",
        "decisive_dimension",
        "schema_exact",
        "correct",
    ]


    results_df[
        reasoning_columns
    ].to_csv(
        paths[
            "reasoning_assessments_csv"
        ],
        index=False,
    )


    confusion_df.to_csv(
        paths[
            "confusion_matrix_csv"
        ]
    )


    family_metrics_df.to_csv(
        paths[
            "family_metrics_csv"
        ],
        index=False,
    )


    variant_metrics_df.to_csv(
        paths[
            "variant_metrics_csv"
        ],
        index=False,
    )


    error_df.to_csv(
        paths[
            "errors_csv"
        ],
        index=False,
    )


    reasoning_inconsistency_df.to_csv(
        paths[
            "reasoning_inconsistencies_csv"
        ],
        index=False,
    )


    assessment_distributions_df.to_csv(
        paths[
            "assessment_distributions_csv"
        ],
        index=False,
    )


    paths[
        "metrics_json"
    ].write_text(
        json.dumps(
            metrics,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


    print("=" * 88)
    print(
        f"{config['experiment_title']} — RESULTS"
    )
    print("=" * 88)

    print(
        "Total cases:",
        metrics[
            "total_cases"
        ],
    )

    print(
        "Valid predictions:",
        metrics[
            "valid_predictions"
        ],
    )

    print(
        "Invalid predictions:",
        metrics[
            "invalid_predictions"
        ],
    )

    print(
        "Exact structured-schema rate:",
        (
            f'{metrics["exact_json_schema_rate"]:.4f}'
        ),
    )

    print(
        "Accuracy:",
        (
            f'{metrics["accuracy_valid_predictions"]:.4f}'
        ),
    )

    print(
        "Balanced accuracy:",
        (
            f'{metrics["balanced_accuracy"]:.4f}'
        ),
    )

    print(
        "ANOMALOUS precision:",
        (
            f'{metrics["anomalous_precision"]:.4f}'
        ),
    )

    print(
        "ANOMALOUS recall:",
        (
            f'{metrics["anomalous_recall"]:.4f}'
        ),
    )

    print(
        "ANOMALOUS F1:",
        (
            f'{metrics["anomalous_f1"]:.4f}'
        ),
    )

    print(
        "NORMAL recall / specificity:",
        (
            f'{metrics["normal_recall_specificity"]:.4f}'
        ),
    )

    print(
        "MCC:",
        (
            f'{metrics["matthews_correlation_coefficient"]:.4f}'
        ),
    )

    print(
        "Matched source-group exact rate:",
        (
            f'{metrics["source_group_exact_match_rate"]:.4f}'
        ),
    )

    print(
        "Reasoning inconsistencies:",
        metrics[
            "reasoning_inconsistency_count"
        ],
    )


    print("\nCONFUSION MATRIX")

    display(
        confusion_df
    )


    print("\nCLASSIFICATION REPORT")

    display(
        classification_report_df
    )


    print("\nPER CASE FAMILY")

    display(
        family_metrics_df
    )


    print("\nPER EXACT CASE VARIANT")

    display(
        variant_metrics_df
    )


    print("\nASSESSMENT DISTRIBUTIONS")

    display(
        assessment_distributions_df
    )


    print(
        "\nSaved cache:",
        paths[
            "prediction_cache"
        ],
    )

    print(
        "Saved prompt:",
        paths[
            "prompt_template"
        ],
    )

    print(
        "Saved prompt diff:",
        paths[
            "prompt_diff"
        ],
    )

    print(
        "Saved predictions:",
        paths[
            "predictions_csv"
        ],
    )

    print(
        "Saved reasoning assessments:",
        paths[
            "reasoning_assessments_csv"
        ],
    )

    print(
        "Saved errors:",
        paths[
            "errors_csv"
        ],
    )


    if len(
        invalid_df
    ):

        print(
            "\nWARNING: Invalid predictions were excluded "
            "from the confusion matrix."
        )


        display(
            invalid_df[
                [
                    "case_id",
                    "case_family",
                    "case_variant",
                    "parse_mode",
                    "raw_output",
                    "generation_error",
                ]
            ]
        )


    else:

        print(
            "\nAll 400 cases produced valid "
            "binary predictions."
        )


    return {
        "metrics": metrics,
        "results_df": results_df,
        "confusion_df": confusion_df,
        "family_metrics_df": family_metrics_df,
        "variant_metrics_df": variant_metrics_df,
        "error_df": error_df,
        "reasoning_inconsistency_df": (
            reasoning_inconsistency_df
        ),
        "assessment_distributions_df": (
            assessment_distributions_df
        ),
    }


# Manual semantic-policy intervention

This section performs no automatic prompt merging.

Run the next cells in order:

1. Print the exact targeted wrong-partner semantic prompt.
2. Print the exact full Structured R1 baseline prompt with semantics, without turns/overlap.
3. Define the complete improved prompt manually.
4. Prepare and inspect the experiment.
5. Run and evaluate all 400 cases.


In [ ]:

# ============================================================
# STRUCTURED R1 — IMPROVED SEMANTIC POLICY
# MANUAL PROMPT WORKFLOW HELPERS
#
# No automatic prompt merging or rewriting is performed.
#
# The notebook prints:
#   1. the exact targeted WRONG_PARTNER semantic prompt,
#   2. the exact full Structured R1 baseline prompt,
#      with semantics but without filtered turns/overlap.
#
# The final combined prompt must then be supplied manually.
#
# Model-facing evidence is fixed to:
#   - participant `speaks` only,
#   - complete local offset distribution,
#   - all five global temporal features,
#   - complete coarse + focused semantic summaries,
#   - no filtered turns,
#   - no overlap.
# ============================================================

from collections import Counter
from IPython.display import display

import copy
import json
import os
import re
import time


IMPROVED_SEMANTIC_BASELINE_EXPERIMENT_NAME = (
    "manual_ablation_l1_no_overlap_evidence_speaks_only"
)

IMPROVED_SEMANTIC_BASELINE_PROMPT_PATH = (
    OUT_DIR
    / IMPROVED_SEMANTIC_BASELINE_EXPERIMENT_NAME
    / "prompt_template.txt"
)

IMPROVED_SEMANTIC_EXPERIMENT_VERSION = (
    "structured_r1_improved_semantic_policy_"
    "speaks_offsets_all_global_no_overlap"
)

IMPROVED_SEMANTIC_EXPERIMENT_TITLE = (
    "Structured R1 — Improved Semantic Policy — "
    "Speaks + Offsets + All Global + Coarse/Focused — No Overlap"
)

IMPROVED_SEMANTIC_ABLATION_ID = (
    "R1_IMPROVED_SEMANTIC_POLICY_"
    "SPEAKS_OFFSETS_ALL_GLOBAL_NO_OVERLAP"
)

IMPROVED_SEMANTIC_INSPECTED_EXPERIMENTS = set()


LOCAL_OVERLAP_FIELDS_IMPROVED = [
    "clean_overlap_seconds",
    "clean_overlap_percent",
]

LOCAL_OFFSET_DISTRIBUTION_FIELDS_IMPROVED = [
    "signed_strict_offsets_seconds",
    "num_signed_strict_offsets",
    "offset_mean_seconds",
    "offset_median_seconds",
    "offset_max_seconds",
    "offset_p75_seconds",
    "offset_p90_seconds",
    "num_offsets_above_1_5_seconds",
    "percent_offsets_above_1_5_seconds",
]

GLOBAL_MODEL_FEATURE_FIELDS_IMPROVED = [
    "best_B_correction_shift_seconds",
    "estimated_B_lateness_seconds",
    "alignment_score_gain_vs_zero",
    "best_num_bilateral_events",
    "best_event_coverage_percent",
]

assert (
    LOCAL_OVERLAP_FIELDS_IMPROVED
    + LOCAL_OFFSET_DISTRIBUTION_FIELDS_IMPROVED
    == LOCAL_FEATURE_FIELDS
)


# ============================================================
# EXACT TARGETED WRONG-PARTNER PROMPT
#
# Extracted from the retained coarse + full-focused semantic-only
# validation experiment. It is printed for manual comparison only.
# ============================================================

TARGETED_WRONG_PARTNER_PROMPT_TEMPLATE = 'You are determining whether two people belong to the SAME real\n120-second dyadic conversation.\n\nYou are given semantic summaries from two synchronized 60-second\nsegments for Participant A and Participant B.\n\nYou must use ONLY the supplied semantic fields.\n\nFor each participant segment, the input contains:\n\nCOARSE SUMMARY:\n- speech_content_summary\n- apparent_topic\n\nFOCUSED SUMMARY:\n- detailed_speech_summary\n- main_topic\n- secondary_topics\n- key_semantic_details\n- summary_specificity\n- unclear_content\n- confidence\n\nThe coarse and focused fields are two independently generated semantic\ndescriptions of the SAME participant segment. Interpret them together.\nDo not treat differences between a participant\'s own coarse and focused\nsummaries as evidence of wrong partner.\n\nDo not use visual engagement, interaction style, speaking amount,\nlistening behavior, or any other information.\n\nThe summaries were independently generated and may sometimes be broad,\nimperfect, or uncertain.\n\nIMPORTANT INTERPRETATION\n\nNORMAL:\n- The two participants\' content can plausibly belong to the same conversation.\n- They may discuss different aspects of a shared subject.\n- One participant\'s content may plausibly respond to, elaborate on, or provide\n  context for the other participant\'s content.\n- A conversation may naturally change topic between Segment 0 and Segment 1.\n- Exact word or topic-label matching is not required.\n\nWRONG_PARTNER:\n- The participants repeatedly discuss unrelated concrete subjects.\n- There is no plausible shared conversational context across the 120 seconds.\n- Both synchronized segments show semantic mismatch, or one segment shows a\n  strong concrete mismatch while the other provides no credible compatibility.\n- The fact that both participants discuss personal experiences, preferences,\n  opinions, daily life, products, or general topics is NOT by itself evidence\n  that they belong to the same conversation.\n\nGENERIC OR WEAK SUMMARIES\n\nBroad labels such as:\n- personal experiences\n- personal preferences\n- general discussion\n- daily life\n- opinions\n- lifestyle\n- personal well-being\n\nmust not be treated as evidence of compatibility unless the concrete content\nalso provides a plausible semantic connection.\n\nIf one segment is vague or generic, treat that segment as\nINSUFFICIENT_EVIDENCE rather than as evidence for NORMAL.\n\nEvaluate the complete 120-second pattern. Do not use a simple vote between\nthe two segments.\n\nReturn ONLY valid JSON with exactly this schema:\n\n{{\n  "label": "NORMAL or ANOMALOUS",\n  "anomaly_type": "none or wrong_partner",\n  "confidence": 0.0,\n  "segment_0_assessment": "compatible / mismatch / insufficient_evidence",\n  "segment_1_assessment": "compatible / mismatch / insufficient_evidence",\n  "cross_segment_assessment": "short assessment of the complete 120-second semantic relationship",\n  "reasoning": "brief evidence-based explanation"\n}}\n\nSEMANTIC INPUT:\n\n{semantic_input}'


WP_DATABASE_SEGMENTS = [
    "segment_0_0_to_60_seconds",
    "segment_1_60_to_120_seconds",
]

WP_COARSE_FIELDS_EXACT = [
    "speech_content_summary",
    "apparent_topic",
]

WP_FOCUSED_FIELDS_EXACT = [
    "detailed_speech_summary",
    "main_topic",
    "secondary_topics",
    "key_semantic_details",
    "summary_specificity",
    "unclear_content",
    "confidence",
]


def build_exact_wp_semantic_input(case):
    combined = {
        "participant_A": {},
        "participant_B": {},
    }

    for role in [
        "participant_A",
        "participant_B",
    ]:
        participant_semantics = case[
            "semantic_summaries"
        ][role]

        for segment_name in WP_DATABASE_SEGMENTS:
            segment_record = participant_semantics[
                segment_name
            ]

            coarse = select_exact_fields(
                segment_record[
                    "coarse_summary"
                ],
                WP_COARSE_FIELDS_EXACT,
                f"{role}/{segment_name}/coarse_summary",
            )

            focused = select_exact_fields(
                segment_record[
                    "focused_summary"
                ],
                WP_FOCUSED_FIELDS_EXACT,
                f"{role}/{segment_name}/focused_summary",
            )

            assert "speaks" not in focused

            combined[role][segment_name] = {
                **coarse,
                "focused_summary": focused,
            }

    return combined


def print_targeted_wrong_partner_prompt(
    *,
    case_index=0,
    prefer_wrong_partner=True,
):
    wp_cases = sorted(
        [
            case
            for case in consolidation_cases
            if get_case_family(case) in {
                "normal",
                "wrong_partner",
            }
        ],
        key=lambda case: str(
            case[
                "case_id"
            ]
        ),
    )

    assert len(wp_cases) == 200

    family_counts = Counter(
        get_case_family(case)
        for case in wp_cases
    )

    assert family_counts == {
        "normal": 100,
        "wrong_partner": 100,
    }

    if prefer_wrong_partner:
        candidate_cases = [
            case
            for case in wp_cases
            if get_case_family(case)
            == "wrong_partner"
        ]
    else:
        candidate_cases = wp_cases

    assert 0 <= case_index < len(candidate_cases)

    case = candidate_cases[
        case_index
    ]

    semantic_input = (
        build_exact_wp_semantic_input(
            case
        )
    )

    rendered_prompt = (
        TARGETED_WRONG_PARTNER_PROMPT_TEMPLATE
        .format(
            semantic_input=json.dumps(
                semantic_input,
                indent=2,
                ensure_ascii=False,
            )
        )
    )

    print("=" * 100)
    print(
        "EXACT TARGETED NORMAL vs WRONG_PARTNER PROMPT"
    )
    print("=" * 100)
    print(
        "Inspection case ID:",
        case[
            "case_id"
        ],
    )
    print(
        "Case metadata is printed outside the model prompt only."
    )
    print(
        "Temporal evidence supplied:",
        False,
    )
    print(
        "Participant speaks supplied:",
        False,
    )
    print(
        "Semantic evidence supplied:",
        "coarse + full focused",
    )
    print("\n" + "=" * 100)
    print("EXACT RENDERED TARGETED PROMPT")
    print("=" * 100)
    print(
        rendered_prompt
    )

    return {
        "case": case,
        "semantic_input": semantic_input,
        "prompt": rendered_prompt,
    }


# ============================================================
# LOAD THE FROZEN FULL STRUCTURED R1 BASELINE PROMPT
#
# Exact L1 No-Overlap baseline:
# speaks + offsets + all global + coarse/focused semantics.
# ============================================================

def load_full_r1_no_overlap_prompt_template():
    assert (
        IMPROVED_SEMANTIC_BASELINE_PROMPT_PATH
        .exists()
    ), (
        "The validated full Structured R1 L1 No-Overlap prompt "
        "was not found:\n"
        f"{IMPROVED_SEMANTIC_BASELINE_PROMPT_PATH}\n\n"
        "Run the L1 configuration/inspection cell in the local "
        "temporal ablation notebook first so prompt_template.txt "
        "is available."
    )

    template = (
        IMPROVED_SEMANTIC_BASELINE_PROMPT_PATH
        .read_text(
            encoding="utf-8"
        )
    )

    required_markers = [
        "Whether each participant speaks",
        "PARTICIPATION VALIDITY",
        "LOCAL TEMPORAL FEATURES",
        "GLOBAL ALIGNMENT-SHIFT FEATURES",
        "SEMANTIC EVIDENCE",
        "STRUCTURED REASONING OUTPUT",
        '"participation_assessment"',
        '"local_temporal_assessment"',
        '"global_temporal_assessment"',
        '"temporal_assessment"',
        '"semantic_assessment"',
        '"decisive_dimension"',
        "{participant_A_speaks}",
        "{participant_B_speaks}",
        "{local_temporal_features}",
        "{global_shift_features}",
        "{semantic_summaries}",
    ]

    for marker in required_markers:
        assert marker in template, (
            "The saved full Structured R1 prompt is missing: "
            f"{marker}"
        )

    forbidden_markers = [
        "Filtered turns:",
        "{participant_A_turns}",
        "{participant_B_turns}",
        "Use the actual filtered turn lists",
        "TURN FORMAT AND BACKCHANNEL FILTERING",
        "clean_overlap_seconds",
        "clean_overlap_percent",
        "FILTERED OVERLAP",
    ]

    template_lower = template.lower()

    for marker in forbidden_markers:
        assert marker.lower() not in template_lower, (
            "Filtered-turn or overlap content exists in the "
            f"saved baseline prompt: {marker}"
        )

    return template


# ============================================================
# FIXED MODEL-FACING INPUT
# ============================================================

def build_improved_semantic_model_input(case):
    payload = copy.deepcopy(
        build_binary_model_input(
            case
        )
    )

    for role in [
        "participant_A",
        "participant_B",
    ]:
        participant_payload = payload[
            role
        ]

        assert "speaks" in participant_payload
        assert "filtered_turns" in participant_payload

        participant_payload.pop(
            "filtered_turns"
        )

        assert set(
            participant_payload
        ) == {
            "speaks"
        }

    local_features = payload[
        "local_temporal_features"
    ]

    for field in (
        LOCAL_OVERLAP_FIELDS_IMPROVED
    ):
        local_features.pop(
            field
        )

    assert set(
        local_features
    ) == set(
        LOCAL_OFFSET_DISTRIBUTION_FIELDS_IMPROVED
    )

    assert set(
        payload[
            "global_shift_features"
        ]
    ) == set(
        GLOBAL_MODEL_FEATURE_FIELDS_IMPROVED
    )

    assert (
        "semantic_summaries"
        in payload
    )

    semantic_keys = (
        collect_nested_keys_reasoning(
            payload[
                "semantic_summaries"
            ]
        )
    )

    for required_key in [
        "speech_content_summary",
        "apparent_topic",
        "detailed_speech_summary",
        "main_topic",
        "secondary_topics",
        "key_semantic_details",
        "summary_specificity",
        "unclear_content",
        "confidence",
    ]:
        assert any(
            key.endswith(
                required_key
            )
            for key in semantic_keys
        ), (
            "Missing semantic field: "
            f"{required_key}"
        )

    return payload


def render_full_r1_prompt_template(
    prompt_template,
    payload,
):
    prompt = prompt_template.format(
        normal_base_reference_text=(
            normal_base_reference_text
        ),
        normal_global_reference_text=(
            normal_global_reference_text
        ),
        duration_seconds=(
            payload[
                "analysis_duration_seconds"
            ]
        ),
        participant_A_speaks=canonical_json(
            payload[
                "participant_A"
            ][
                "speaks"
            ]
        ),
        participant_B_speaks=canonical_json(
            payload[
                "participant_B"
            ][
                "speaks"
            ]
        ),
        local_temporal_features=json.dumps(
            payload[
                "local_temporal_features"
            ],
            indent=2,
            ensure_ascii=False,
        ),
        global_shift_features=json.dumps(
            payload[
                "global_shift_features"
            ],
            indent=2,
            ensure_ascii=False,
        ),
        semantic_summaries=json.dumps(
            payload[
                "semantic_summaries"
            ],
            indent=2,
            ensure_ascii=False,
        ),
    )

    prompt_lower = prompt.lower()

    for forbidden_key in FORBIDDEN_PROMPT_KEYS:
        assert (
            forbidden_key.lower()
            not in prompt_lower
        )

    for forbidden_marker in [
        "filtered turns:",
        "participant_a_filtered_turns",
        "participant_b_filtered_turns",
        "clean_overlap_seconds",
        "clean_overlap_percent",
        "filtered overlap",
    ]:
        assert (
            forbidden_marker
            not in prompt_lower
        )

    return prompt


def print_full_r1_no_overlap_prompt(
    *,
    case_index=0,
):
    ordered_cases = sorted(
        consolidation_cases,
        key=lambda case: str(
            case[
                "case_id"
            ]
        ),
    )

    assert (
        0
        <= case_index
        < len(
            ordered_cases
        )
    )

    case = ordered_cases[
        case_index
    ]

    template = (
        load_full_r1_no_overlap_prompt_template()
    )

    payload = (
        build_improved_semantic_model_input(
            case
        )
    )

    prompt = (
        render_full_r1_prompt_template(
            template,
            payload,
        )
    )

    export_dir = (
        OUT_DIR
        / "improved_semantic_policy_manual_prompts"
    )

    export_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    export_path = (
        export_dir
        / "full_structured_r1_no_overlap_rendered_prompt.txt"
    )

    export_path.write_text(
        prompt,
        encoding="utf-8",
    )

    print("=" * 100)
    print(
        "EXACT FULL STRUCTURED R1 BASELINE PROMPT"
    )
    print(
        "Speaks + Offsets + All Global + Coarse/Focused"
    )
    print(
        "No Filtered Turns · No Overlap"
    )
    print("=" * 100)
    print(
        "Inspection case ID:",
        case[
            "case_id"
        ],
    )
    print(
        "Filtered turns supplied:",
        False,
    )
    print(
        "Overlap evidence supplied:",
        False,
    )
    print(
        "Semantic evidence supplied:",
        "coarse + focused",
    )
    print("\nEXACT MODEL-FACING PAYLOAD")
    print(
        json.dumps(
            payload,
            indent=2,
            ensure_ascii=False,
        )
    )
    print("\n" + "=" * 100)
    print("EXACT RENDERED STRUCTURED R1 PROMPT")
    print("=" * 100)
    print(
        prompt
    )
    print("\nSaved rendered prompt:", export_path)

    return {
        "case": case,
        "payload": payload,
        "prompt": prompt,
        "template": template,
        "export_path": export_path,
    }


# ============================================================
# MANUAL COMBINED-PROMPT RENDERER
# ============================================================

def render_manual_improved_semantic_prompt(
    prompt_template,
    payload,
):
    assert isinstance(
        prompt_template,
        str,
    )

    assert (
        prompt_template.strip()
    )

    prompt = prompt_template.format(
        normal_base_reference_text=(
            normal_base_reference_text
        ),
        normal_global_reference_text=(
            normal_global_reference_text
        ),
        duration_seconds=(
            payload[
                "analysis_duration_seconds"
            ]
        ),
        participant_A_speaks=canonical_json(
            payload[
                "participant_A"
            ][
                "speaks"
            ]
        ),
        participant_B_speaks=canonical_json(
            payload[
                "participant_B"
            ][
                "speaks"
            ]
        ),
        local_temporal_features=json.dumps(
            payload[
                "local_temporal_features"
            ],
            indent=2,
            ensure_ascii=False,
        ),
        global_shift_features=json.dumps(
            payload[
                "global_shift_features"
            ],
            indent=2,
            ensure_ascii=False,
        ),
        semantic_summaries=json.dumps(
            payload[
                "semantic_summaries"
            ],
            indent=2,
            ensure_ascii=False,
        ),
    )

    prompt_lower = (
        prompt.lower()
    )

    for forbidden_key in FORBIDDEN_PROMPT_KEYS:
        assert (
            forbidden_key.lower()
            not in prompt_lower
        ), (
            "Forbidden field leaked into prompt: "
            f"{forbidden_key}"
        )

    for forbidden_marker in [
        "filtered turns:",
        "participant_a_filtered_turns",
        "participant_b_filtered_turns",
        "{participant_a_turns}",
        "{participant_b_turns}",
        "clean_overlap_seconds",
        "clean_overlap_percent",
        "filtered overlap",
    ]:
        assert (
            forbidden_marker
            not in prompt_lower
        ), (
            "Forbidden turns/overlap marker: "
            f"{forbidden_marker}"
        )

    return prompt


# ============================================================
# MANUAL PROMPT VALIDATOR
# ============================================================

def validate_manual_improved_semantic_prompt(
    prompt_template,
):
    assert isinstance(
        prompt_template,
        str,
    ), (
        "Set R1_IMPROVED_SEMANTIC_MANUAL_PROMPT_TEMPLATE "
        "to one complete prompt string."
    )

    template = (
        prompt_template.strip()
    )

    assert template

    required_placeholders = [
        "{duration_seconds}",
        "{participant_A_speaks}",
        "{participant_B_speaks}",
        "{local_temporal_features}",
        "{global_shift_features}",
        "{semantic_summaries}",
    ]

    for placeholder in required_placeholders:
        assert placeholder in template, (
            "Missing placeholder: "
            f"{placeholder}"
        )

    required_sections = [
        "PARTICIPATION VALIDITY",
        "LOCAL TEMPORAL FEATURES",
        "GLOBAL ALIGNMENT-SHIFT FEATURES",
        "SEMANTIC EVIDENCE",
        "FINAL COMBINED DECISION",
        "CURRENT CASE",
        "STRUCTURED REASONING OUTPUT",
        "OUTPUT",
    ]

    for marker in required_sections:
        assert marker in template, (
            "Missing required section: "
            f"{marker}"
        )

    required_schema_markers = [
        '"participation_assessment"',
        '"local_temporal_assessment"',
        '"global_temporal_assessment"',
        '"temporal_assessment"',
        '"semantic_assessment"',
        '"decisive_dimension"',
        '"label"',
        "COMPATIBLE",
        "INCOMPATIBLE",
        "LIMITED",
        "PARTICIPATION",
        "TEMPORAL",
        "SEMANTIC",
        "NONE",
    ]

    for marker in required_schema_markers:
        assert marker in template, (
            "Missing structured-output marker: "
            f"{marker}"
        )

    semantic_policy_markers = [
        "plausible",
        "concrete",
        "insufficient",
        "generic",
        "same conversation",
    ]

    template_lower = (
        template.lower()
    )

    for marker in semantic_policy_markers:
        assert marker in template_lower, (
            "The manual semantic policy appears incomplete. "
            f"Missing concept: {marker}"
        )

    forbidden_markers = [
        "filtered turns:",
        "{participant_A_turns}",
        "{participant_B_turns}",
        "TURN FORMAT AND BACKCHANNEL FILTERING",
        "clean_overlap_seconds",
        "clean_overlap_percent",
        "FILTERED OVERLAP",
    ]

    for marker in forbidden_markers:
        assert marker.lower() not in template_lower, (
            "Filtered-turn or overlap content remains: "
            f"{marker}"
        )

    return template


def build_manual_improved_semantic_prompt(
    case,
    config,
):
    payload = (
        build_improved_semantic_model_input(
            case
        )
    )

    prompt = (
        render_manual_improved_semantic_prompt(
            config[
                "reasoning_prompt_template"
            ],
            payload,
        )
    )

    return (
        prompt,
        payload,
    )


# ============================================================
# PREPARE CONFIG
# ============================================================

def prepare_manual_improved_semantic_experiment(
    *,
    manual_prompt_template,
):
    prompt_template = (
        validate_manual_improved_semantic_prompt(
            manual_prompt_template
        )
    )

    baseline_template = (
        load_full_r1_no_overlap_prompt_template()
    )

    experiment_dir = (
        OUT_DIR
        / IMPROVED_SEMANTIC_EXPERIMENT_VERSION
    )

    experiment_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    paths = {
        "prediction_cache": (
            experiment_dir
            / "predictions_cache.json"
        ),
        "predictions_csv": (
            experiment_dir
            / "predictions_all_400.csv"
        ),
        "metrics_json": (
            experiment_dir
            / "metrics.json"
        ),
        "confusion_matrix_csv": (
            experiment_dir
            / "confusion_matrix.csv"
        ),
        "family_metrics_csv": (
            experiment_dir
            / "metrics_by_case_family.csv"
        ),
        "variant_metrics_csv": (
            experiment_dir
            / "metrics_by_case_variant.csv"
        ),
        "errors_csv": (
            experiment_dir
            / "classification_errors.csv"
        ),
        "reasoning_assessments_csv": (
            experiment_dir
            / "reasoning_assessments.csv"
        ),
        "reasoning_inconsistencies_csv": (
            experiment_dir
            / "reasoning_inconsistencies.csv"
        ),
        "assessment_distributions_csv": (
            experiment_dir
            / "assessment_distributions.csv"
        ),
        "prompt_template": (
            experiment_dir
            / "prompt_template.txt"
        ),
        "baseline_prompt_copy": (
            experiment_dir
            / "baseline_prompt_template.txt"
        ),
        "targeted_wp_prompt_copy": (
            experiment_dir
            / "targeted_wrong_partner_prompt_template.txt"
        ),
        "experiment_manifest": (
            experiment_dir
            / "experiment_manifest.json"
        ),
    }

    reasoning_prompt_sha256 = (
        sha256_text(
            prompt_template
        )
    )

    baseline_prompt_sha256 = (
        sha256_text(
            baseline_template
        )
    )

    targeted_wp_prompt_sha256 = (
        sha256_text(
            TARGETED_WRONG_PARTNER_PROMPT_TEMPLATE
        )
    )

    normal_reference_sha256 = (
        sha256_text(
            normal_base_reference_text
            + "\n"
            + normal_global_reference_text
        )
    )

    paths[
        "prompt_template"
    ].write_text(
        prompt_template,
        encoding="utf-8",
    )

    paths[
        "baseline_prompt_copy"
    ].write_text(
        baseline_template,
        encoding="utf-8",
    )

    paths[
        "targeted_wp_prompt_copy"
    ].write_text(
        TARGETED_WRONG_PARTNER_PROMPT_TEMPLATE,
        encoding="utf-8",
    )

    example_prompt, example_payload = (
        build_manual_improved_semantic_prompt(
            consolidation_cases[0],
            {
                "reasoning_prompt_template": (
                    prompt_template
                )
            },
        )
    )

    manifest = {
        "experiment_version": (
            IMPROVED_SEMANTIC_EXPERIMENT_VERSION
        ),
        "experiment_title": (
            IMPROVED_SEMANTIC_EXPERIMENT_TITLE
        ),
        "ablation_id": (
            IMPROVED_SEMANTIC_ABLATION_ID
        ),
        "source_experiment": (
            IMPROVED_SEMANTIC_BASELINE_EXPERIMENT_NAME
        ),
        "model_id": MODEL_ID,
        "max_new_tokens": (
            MAX_NEW_TOKENS_REASONING
        ),
        "reasoning_prompt_sha256": (
            reasoning_prompt_sha256
        ),
        "baseline_prompt_sha256": (
            baseline_prompt_sha256
        ),
        "targeted_wp_prompt_sha256": (
            targeted_wp_prompt_sha256
        ),
        "normal_reference_sha256": (
            normal_reference_sha256
        ),
        "semantic_input": (
            "coarse_and_focused"
        ),
        "focused_summaries_used": True,
        "filtered_turns_supplied": False,
        "overlap_supplied": False,
        "participant_model_input": (
            "speaks_only"
        ),
        "local_model_input": (
            "complete_offset_distribution_only"
        ),
        "global_model_input": (
            "all_five_global_features"
        ),
        "temporal_profiles_used": (
            "frozen_NORMAL_only"
        ),
        "assessment_policy": (
            "structured_R1_with_targeted_semantic_policy"
        ),
        "schema_keys": list(
            REASONING_SCHEMA_KEYS
        ),
        "prompt_definition_method": (
            "manual_complete_template"
        ),
        "decoding": {
            "do_sample": False,
            "use_cache": True,
        },
    }

    paths[
        "experiment_manifest"
    ].write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    config = {
        **manifest,
        "experiment_dir": (
            experiment_dir
        ),
        "reasoning_prompt_template": (
            prompt_template
        ),
        "paths": paths,
    }

    print("=" * 100)
    print(
        "IMPROVED SEMANTIC POLICY — CONFIGURATION READY"
    )
    print("=" * 100)
    print(
        "Experiment version:",
        config[
            "experiment_version"
        ],
    )
    print(
        "Cases:",
        len(
            consolidation_cases
        ),
    )
    print(
        "Semantic input:",
        config[
            "semantic_input"
        ],
    )
    print(
        "Filtered turns supplied:",
        False,
    )
    print(
        "Overlap supplied:",
        False,
    )
    print(
        "Prompt SHA256:",
        reasoning_prompt_sha256,
    )
    print(
        "Example prompt characters:",
        len(
            example_prompt
        ),
    )
    print(
        "Prediction cache:",
        paths[
            "prediction_cache"
        ],
    )
    print(
        "Existing cache:",
        paths[
            "prediction_cache"
        ].exists(),
    )

    return config


# ============================================================
# REQUIRED INSPECTION
# ============================================================

def inspect_manual_improved_semantic_prompt(
    config,
    *,
    case_index=0,
):
    ordered_cases = sorted(
        consolidation_cases,
        key=lambda case: str(
            case[
                "case_id"
            ]
        ),
    )

    assert (
        0
        <= case_index
        < len(
            ordered_cases
        )
    )

    case = ordered_cases[
        case_index
    ]

    prompt, payload = (
        build_manual_improved_semantic_prompt(
            case,
            config,
        )
    )

    print("=" * 100)
    print(
        "MANUAL PROMPT INSPECTION — "
        "STRUCTURED R1 IMPROVED SEMANTIC POLICY"
    )
    print("=" * 100)
    print(
        "Inspection case ID:",
        case[
            "case_id"
        ],
    )
    print(
        "Gold label is intentionally NOT printed inside the model prompt."
    )
    print(
        "Participant evidence supplied:",
        "speaks only",
    )
    print(
        "Local evidence supplied:",
        "complete offset distribution only",
    )
    print(
        "Global evidence supplied:",
        "all five global temporal features",
    )
    print(
        "Semantic evidence supplied:",
        "coarse + focused",
    )
    print(
        "Filtered turns supplied:",
        False,
    )
    print(
        "Overlap supplied:",
        False,
    )
    print("\n" + "=" * 100)
    print("EXACT MODEL-FACING PAYLOAD")
    print("=" * 100)
    print(
        json.dumps(
            payload,
            indent=2,
            ensure_ascii=False,
        )
    )
    print("\n" + "=" * 100)
    print("EXACT RENDERED MODEL PROMPT")
    print("=" * 100)
    print(
        prompt
    )

    IMPROVED_SEMANTIC_INSPECTED_EXPERIMENTS.add(
        config[
            "experiment_version"
        ]
    )

    print(
        "\nInspection unlocked:",
        config[
            "experiment_version"
        ]
        in IMPROVED_SEMANTIC_INSPECTED_EXPERIMENTS,
    )

    return {
        "case": case,
        "payload": payload,
        "prompt": prompt,
    }


# ============================================================
# CACHE
# ============================================================

def create_improved_semantic_cache(
    config,
):
    return {
        "experiment_version": (
            config[
                "experiment_version"
            ]
        ),
        "experiment_title": (
            config[
                "experiment_title"
            ]
        ),
        "ablation_id": (
            config[
                "ablation_id"
            ]
        ),
        "source_experiment": (
            config[
                "source_experiment"
            ]
        ),
        "model_id": (
            config[
                "model_id"
            ]
        ),
        "reasoning_prompt_sha256": (
            config[
                "reasoning_prompt_sha256"
            ]
        ),
        "baseline_prompt_sha256": (
            config[
                "baseline_prompt_sha256"
            ]
        ),
        "targeted_wp_prompt_sha256": (
            config[
                "targeted_wp_prompt_sha256"
            ]
        ),
        "normal_reference_sha256": (
            config[
                "normal_reference_sha256"
            ]
        ),
        "semantic_input": (
            config[
                "semantic_input"
            ]
        ),
        "focused_summaries_used": True,
        "filtered_turns_supplied": False,
        "overlap_supplied": False,
        "temporal_profiles_used": (
            config[
                "temporal_profiles_used"
            ]
        ),
        "assessment_policy": (
            config[
                "assessment_policy"
            ]
        ),
        "schema_keys": list(
            REASONING_SCHEMA_KEYS
        ),
        "max_new_tokens": (
            MAX_NEW_TOKENS_REASONING
        ),
        "created_at_utc": (
            reasoning_utc_now()
        ),
        "updated_at_utc": (
            reasoning_utc_now()
        ),
        "records": {},
    }


# ============================================================
# INFERENCE
# ============================================================

def run_manual_improved_semantic_experiment(
    config,
    *,
    print_each_case_prompt=False,
):
    assert (
        config[
            "experiment_version"
        ]
        in IMPROVED_SEMANTIC_INSPECTED_EXPERIMENTS
    ), (
        "Run the exact manual prompt-inspection cell "
        "before inference."
    )

    prediction_cache_path = (
        config[
            "paths"
        ][
            "prediction_cache"
        ]
    )

    expected_cache = (
        create_improved_semantic_cache(
            config
        )
    )

    if prediction_cache_path.exists():
        prediction_cache = json.loads(
            prediction_cache_path.read_text(
                encoding="utf-8"
            )
        )

        for key in [
            "experiment_version",
            "ablation_id",
            "source_experiment",
            "model_id",
            "reasoning_prompt_sha256",
            "baseline_prompt_sha256",
            "targeted_wp_prompt_sha256",
            "normal_reference_sha256",
            "semantic_input",
            "focused_summaries_used",
            "filtered_turns_supplied",
            "overlap_supplied",
            "temporal_profiles_used",
            "assessment_policy",
            "schema_keys",
            "max_new_tokens",
        ]:
            assert (
                prediction_cache[
                    key
                ]
                == expected_cache[
                    key
                ]
            ), (
                "Cache mismatch for "
                f"{key}"
            )

        print(
            "Resuming cache:",
            prediction_cache_path,
        )
        print(
            "Existing records:",
            len(
                prediction_cache[
                    "records"
                ]
            ),
        )

    else:
        prediction_cache = expected_cache

        reasoning_atomic_write_json(
            prediction_cache_path,
            prediction_cache,
        )

        print(
            "Created cache:",
            prediction_cache_path,
        )

    ordered_cases = sorted(
        consolidation_cases,
        key=lambda case: str(
            case[
                "case_id"
            ]
        ),
    )

    assert len(
        ordered_cases
    ) == 400

    for case in tqdm(
        ordered_cases,
        desc=(
            config[
                "experiment_version"
            ]
        ),
    ):
        case_id = str(
            case[
                "case_id"
            ]
        )

        prompt, payload = (
            build_manual_improved_semantic_prompt(
                case,
                config,
            )
        )

        if print_each_case_prompt:
            print("\n" + "=" * 100)
            print("CASE:", case_id)
            print("=" * 100)
            print(prompt)

        prompt_sha256 = (
            sha256_text(
                prompt
            )
        )

        payload_sha256 = (
            sha256_text(
                canonical_json(
                    payload
                )
            )
        )

        existing_record = (
            prediction_cache[
                "records"
            ].get(
                case_id
            )
        )

        if (
            existing_record is not None
            and existing_record.get(
                "prediction"
            ) in LABELS
        ):
            assert (
                existing_record[
                    "prompt_sha256"
                ]
                == prompt_sha256
            )
            assert (
                existing_record[
                    "input_payload_sha256"
                ]
                == payload_sha256
            )
            continue

        started = (
            time.perf_counter()
        )

        try:
            raw_output, input_token_count = (
                qwen_text_only_binary(
                    prompt,
                    max_new_tokens=(
                        MAX_NEW_TOKENS_REASONING
                    ),
                )
            )

            parsed_result = (
                parse_structured_reasoning_prediction(
                    raw_output
                )
            )

            generation_error = None

        except Exception as exc:
            raw_output = ""
            input_token_count = None

            parsed_result = {
                "prediction": None,
                "parse_mode": (
                    "generation_error"
                ),
                "schema_exact": False,
                "parsed_output": None,
                "schema_errors": [],
                "participation_assessment": None,
                "local_temporal_assessment": None,
                "global_temporal_assessment": None,
                "temporal_assessment": None,
                "semantic_assessment": None,
                "decisive_dimension": None,
            }

            generation_error = (
                f"{type(exc).__name__}: {exc}"
            )

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        elapsed_seconds = (
            time.perf_counter()
            - started
        )

        prediction_cache[
            "records"
        ][case_id] = {
            "case_id": case_id,
            "source_group_id": str(
                case[
                    "source_group_id"
                ]
            ),
            "case_family": (
                get_case_family(
                    case
                )
            ),
            "case_variant": str(
                case[
                    "case_variant"
                ]
            ),
            "gold_binary_label": str(
                case[
                    "gold_binary_label"
                ]
            ).upper(),
            "prompt_sha256": (
                prompt_sha256
            ),
            "input_payload_sha256": (
                payload_sha256
            ),
            "semantic_input": (
                "coarse_and_focused"
            ),
            "focused_summaries_used": True,
            "input_token_count": (
                input_token_count
            ),
            "max_new_tokens": (
                MAX_NEW_TOKENS_REASONING
            ),
            "raw_output": (
                raw_output
            ),
            "prediction": (
                parsed_result[
                    "prediction"
                ]
            ),
            "participation_assessment": (
                parsed_result[
                    "participation_assessment"
                ]
            ),
            "local_temporal_assessment": (
                parsed_result[
                    "local_temporal_assessment"
                ]
            ),
            "global_temporal_assessment": (
                parsed_result[
                    "global_temporal_assessment"
                ]
            ),
            "temporal_assessment": (
                parsed_result[
                    "temporal_assessment"
                ]
            ),
            "semantic_assessment": (
                parsed_result[
                    "semantic_assessment"
                ]
            ),
            "decisive_dimension": (
                parsed_result[
                    "decisive_dimension"
                ]
            ),
            "parse_mode": (
                parsed_result[
                    "parse_mode"
                ]
            ),
            "schema_exact": bool(
                parsed_result[
                    "schema_exact"
                ]
            ),
            "schema_errors": (
                parsed_result[
                    "schema_errors"
                ]
            ),
            "parsed_output": (
                parsed_result[
                    "parsed_output"
                ]
            ),
            "generation_error": (
                generation_error
            ),
            "elapsed_seconds": round(
                elapsed_seconds,
                4,
            ),
            "completed_at_utc": (
                reasoning_utc_now()
            ),
        }

        prediction_cache[
            "updated_at_utc"
        ] = reasoning_utc_now()

        reasoning_atomic_write_json(
            prediction_cache_path,
            prediction_cache,
        )

    print("\n" + "=" * 100)
    print(
        "IMPROVED SEMANTIC POLICY — INFERENCE COMPLETE"
    )
    print("=" * 100)
    print(
        "Cached records:",
        len(
            prediction_cache[
                "records"
            ]
        ),
    )
    print(
        "Prediction cache:",
        prediction_cache_path,
    )

    return prediction_cache


# Required manual workflow

In [ ]:
# STEP 1 — PRINT THE EXACT TARGETED WRONG-PARTNER PROMPT

TARGETED_WRONG_PARTNER_INSPECTION = (
    print_targeted_wrong_partner_prompt(
        case_index=0,
        prefer_wrong_partner=True,
    )
)


EXACT TARGETED NORMAL vs WRONG_PARTNER PROMPT
Inspection case ID: consolidation_wrong_partner_000
Case metadata is printed outside the model prompt only.
Temporal evidence supplied: False
Participant speaks supplied: False
Semantic evidence supplied: coarse + full focused

EXACT RENDERED TARGETED PROMPT
You are determining whether two people belong to the SAME real
120-second dyadic conversation.

You are given semantic summaries from two synchronized 60-second
segments for Participant A and Participant B.

You must use ONLY the supplied semantic fields.

For each participant segment, the input contains:

COARSE SUMMARY:
- speech_content_summary
- apparent_topic

FOCUSED SUMMARY:
- detailed_speech_summary
- main_topic
- secondary_topics
- key_semantic_details
- summary_specificity
- unclear_content
- confidence

The coarse and focused fields are two independently generated semantic
descriptions of the SAME participant segment. Interpret them together.
Do not treat differences between a

In [ ]:
# STEP 2 — PRINT THE EXACT FULL STRUCTURED R1 BASELINE PROMPT
# This baseline already contains:
# - speaks only,
# - local offsets only,
# - all five global temporal features,
# - coarse + focused semantics,
# - no filtered turns,
# - no overlap.

FULL_R1_BASELINE_INSPECTION = (
    print_full_r1_no_overlap_prompt(
        case_index=0,
    )
)


EXACT FULL STRUCTURED R1 BASELINE PROMPT
Speaks + Offsets + All Global + Coarse/Focused
No Filtered Turns · No Overlap
Inspection case ID: consolidation_lag_2sec_000
Filtered turns supplied: False
Overlap evidence supplied: False
Semantic evidence supplied: coarse + focused

EXACT MODEL-FACING PAYLOAD
{
  "analysis_duration_seconds": 120.0,
  "participant_A": {
    "speaks": true
  },
  "participant_B": {
    "speaks": true
  },
  "local_temporal_features": {
    "signed_strict_offsets_seconds": [
      3.16,
      4.28
    ],
    "num_signed_strict_offsets": 2,
    "offset_mean_seconds": 3.72,
    "offset_median_seconds": 3.72,
    "offset_max_seconds": 4.28,
    "offset_p75_seconds": 4.0,
    "offset_p90_seconds": 4.17,
    "num_offsets_above_1_5_seconds": 2,
    "percent_offsets_above_1_5_seconds": 100.0
  },
  "global_shift_features": {
    "best_B_correction_shift_seconds": 2.7,
    "estimated_B_lateness_seconds": 0.0,
    "alignment_score_gain_vs_zero": 0.069487,
    "best_num_bi

In [ ]:
# STEP 3 — MANUALLY DEFINE THE COMPLETE COMBINED PROMPT
#
# Do not generate this automatically.
#
# Start from the exact full Structured R1 prompt printed above.
# Keep its participation, local temporal, global temporal,
# final-decision and structured-output logic fixed.
#
# Replace only the semantic reasoning policy with a carefully
# integrated version of the targeted wrong-partner logic.
#
# The final prompt must retain all placeholders:
#   {duration_seconds}
#   {participant_A_speaks}
#   {participant_B_speaks}
#   {local_temporal_features}
#   {global_shift_features}
#   {semantic_summaries}

# ============================================================
# STEP 3 — MANUALLY DEFINE THE IMPROVED SEMANTIC R1 PROMPT
#
# Surgical intervention:
# - keep the complete Structured R1 baseline unchanged
# - replace only the SEMANTIC EVIDENCE policy
# ============================================================

import difflib


# Load the exact full Structured R1 baseline template:
# - speaks only
# - complete local offset distribution
# - all five global features
# - coarse + focused semantics
# - no filtered turns
# - no overlap
ORIGINAL_FULL_R1_PROMPT_TEMPLATE = (
    load_full_r1_no_overlap_prompt_template()
)


SEMANTIC_SECTION_HEADER = """
============================================================
SEMANTIC EVIDENCE
============================================================
""".strip()


FINAL_DECISION_SECTION_HEADER = """
============================================================
FINAL COMBINED DECISION
============================================================
""".strip()


# ============================================================
# VERIFY THAT THE TWO SECTION BOUNDARIES ARE UNIQUE
# ============================================================

assert (
    ORIGINAL_FULL_R1_PROMPT_TEMPLATE.count(
        SEMANTIC_SECTION_HEADER
    )
    == 1
), (
    "Expected exactly one SEMANTIC EVIDENCE section."
)


assert (
    ORIGINAL_FULL_R1_PROMPT_TEMPLATE.count(
        FINAL_DECISION_SECTION_HEADER
    )
    == 1
), (
    "Expected exactly one FINAL COMBINED DECISION section."
)


(
    PROMPT_BEFORE_SEMANTIC,
    semantic_header_found,
    PROMPT_FROM_SEMANTIC_ONWARD,
) = ORIGINAL_FULL_R1_PROMPT_TEMPLATE.partition(
    SEMANTIC_SECTION_HEADER
)


assert semantic_header_found == SEMANTIC_SECTION_HEADER


(
    ORIGINAL_SEMANTIC_POLICY_BODY,
    final_decision_header_found,
    PROMPT_AFTER_FINAL_DECISION_HEADER,
) = PROMPT_FROM_SEMANTIC_ONWARD.partition(
    FINAL_DECISION_SECTION_HEADER
)


assert (
    final_decision_header_found
    == FINAL_DECISION_SECTION_HEADER
)


ORIGINAL_SEMANTIC_POLICY_BODY = (
    ORIGINAL_SEMANTIC_POLICY_BODY.strip()
)


# ============================================================
# IMPROVED SEMANTIC POLICY
#
# Adapted from the successful targeted wrong-partner pipeline.
#
# Crucially:
# - semantic_assessment uses semantic summaries only
# - coarse and focused summaries are interpreted together
# - specific unrelated content becomes INCOMPATIBLE
# - vague content becomes LIMITED, not COMPATIBLE
# - temporal and participation evidence cannot influence the
#   semantic_assessment
# ============================================================

IMPROVED_SEMANTIC_POLICY_BODY = r"""
The 120-second interval is divided into:

- Segment 0: 0 to 60 seconds
- Segment 1: 60 to 120 seconds

For each participant and synchronized segment, you receive:

COARSE SUMMARY:

- speech_content_summary
- apparent_topic

FOCUSED SUMMARY:

- detailed_speech_summary
- main_topic
- secondary_topics
- key_semantic_details
- summary_specificity
- unclear_content
- confidence

The coarse and focused fields are two independently generated semantic
descriptions of the SAME participant segment.

Interpret the coarse and focused fields together.

Do not treat differences between one participant's own coarse and
focused summaries as evidence of semantic incompatibility.

The summaries were independently generated and may sometimes be broad,
imperfect, repetitive, incomplete, or uncertain.

The semantic assessment must be based ONLY on the supplied semantic
summaries.

When assigning semantic_assessment:

- do not use the participation evidence,
- do not use the local temporal evidence,
- do not use the global temporal evidence,
- do not use whether the temporal pattern appears NORMAL or ANOMALOUS,
- and do not allow another reasoning branch to soften or strengthen
  the semantic assessment.

The semantic assessment must independently answer:

Can the two participants' supplied content plausibly belong to the same
real 120-second dyadic conversation?

Internally evaluate:

1. The semantic relationship between Participant A and Participant B
   in Segment 0.
2. The semantic relationship between Participant A and Participant B
   in Segment 1.
3. The complete cross-segment semantic relationship across the full
   120 seconds.

Do not expose these three internal checks as additional output fields.

Do not use a simple vote between Segment 0 and Segment 1.

Evaluate the complete 120-second semantic pattern.

============================================================
SEMANTIC COMPATIBILITY
============================================================

Semantic evidence is COMPATIBLE when the two participants' content can
plausibly belong to the same conversation.

A compatible relationship may include:

- a shared concrete subject,
- compatible people, events, places, experiences, or arguments,
- complementary accounts of the same situation,
- a plausible question-and-response relationship,
- one participant responding to or reacting to the other,
- one participant supplying context for the other,
- one participant elaborating on a detail introduced by the other,
- different aspects of one shared subject,
- or a coherent topic transition between Segment 0 and Segment 1.

The participants do not need to use identical words.

Exact topic-label matching is not required.

The participants may describe different parts of the same event or
story.

One participant may provide information that is not repeated by the
other participant, provided that the information still fits a plausible
shared conversational context.

A real conversation may naturally change topic during the 120-second
interval.

A topic change does not by itself establish semantic incompatibility.

Assign:

semantic_assessment = COMPATIBLE

only when the concrete content provides credible evidence that the two
participants can plausibly belong to the same conversation.

============================================================
SEMANTIC INCOMPATIBILITY
============================================================

Semantic evidence is INCOMPATIBLE when the participant summaries contain
specific, concrete content that does not support one plausible shared
conversation.

Strong semantic incompatibility may include:

- the participants repeatedly discussing unrelated concrete subjects,
- incompatible people, events, places, activities, products, stories,
  situations, or arguments,
- one participant describing a concrete subject while the other
  describes a clearly unrelated concrete subject,
- both synchronized segments showing semantic mismatch,
- or one synchronized segment showing a strong concrete mismatch while
  the other segment provides no credible evidence of compatibility.

Specific but unrelated content is INCOMPATIBLE.

It must not be treated as LIMITED merely because the model cannot find
a connection.

Do not invent an indirect relationship merely to make the participants
compatible.

Do not assume that two unrelated stories belong to the same conversation
only because both are personal stories.

Do not assume compatibility only because both participants discuss:

- personal experiences,
- personal preferences,
- opinions,
- daily life,
- family,
- relationships,
- products,
- health,
- lifestyle,
- personal well-being,
- or other broad categories.

The fact that two summaries fall under the same broad category is not
sufficient evidence that they describe the same conversation.

Assign:

semantic_assessment = INCOMPATIBLE

when the concrete semantic evidence strongly indicates that the two
participants do not belong to one plausible shared conversation.

============================================================
GENERIC, WEAK, OR UNCERTAIN SEMANTIC EVIDENCE
============================================================

Broad or generic labels such as:

- personal experiences
- personal preferences
- general discussion
- daily life
- opinions
- lifestyle
- personal well-being
- family matters
- emotional topics

must not be treated as evidence of compatibility unless the concrete
summary content also provides a plausible semantic connection.

If one synchronized segment is vague, generic, unclear, incomplete, or
low-confidence, treat that segment internally as insufficient evidence.

An insufficient segment must not automatically support COMPATIBLE.

However, one insufficient segment does not automatically require the
complete semantic assessment to be LIMITED.

Use the other synchronized segment and the complete 120-second pattern:

- If the available concrete evidence clearly supports one plausible
  shared conversation, assign COMPATIBLE.
- If the available concrete evidence shows a strong mismatch and the
  weak segment provides no credible compatibility, assign INCOMPATIBLE.
- If the available summaries are too vague, weak, contradictory, or
  uncertain to establish either compatibility or incompatibility,
  assign LIMITED.

Assign:

semantic_assessment = LIMITED

only when the semantic evidence is genuinely insufficient to determine
whether the two participants belong to the same conversation.

Do not use LIMITED for specific but unrelated content.

============================================================
SEMANTIC ASSESSMENT RULE
============================================================

Use exactly these meanings:

- COMPATIBLE:
  The concrete content supports a plausible shared conversation.

- INCOMPATIBLE:
  The concrete content provides strong evidence of unrelated participant
  records that do not plausibly form one conversation.

- LIMITED:
  The summaries are too generic, unclear, incomplete, contradictory, or
  uncertain to support either conclusion reliably.

The semantic assessment is an independent diagnostic assessment.

Determine it from the semantic summaries before applying the final
combined decision policy.

Semantic COMPATIBLE must not cancel reliable temporal failure.

Semantic INCOMPATIBLE must not be cancelled by apparently normal temporal
coordination.

Semantic LIMITED means that the semantic branch is inconclusive; it does
not itself establish either NORMAL or ANOMALOUS.
""".strip()


# ============================================================
# CONSTRUCT THE COMPLETE PROMPT
#
# Everything before and after SEMANTIC EVIDENCE remains exactly
# as it was in the original full Structured R1 baseline.
# ============================================================

R1_IMPROVED_SEMANTIC_MANUAL_PROMPT_TEMPLATE = (
    PROMPT_BEFORE_SEMANTIC
    + SEMANTIC_SECTION_HEADER
    + "\n\n"
    + IMPROVED_SEMANTIC_POLICY_BODY
    + "\n\n"
    + FINAL_DECISION_SECTION_HEADER
    + PROMPT_AFTER_FINAL_DECISION_HEADER
).strip()


# ============================================================
# AUDIT: VERIFY THAT ONLY THE SEMANTIC SECTION CHANGED
# ============================================================

assert (
    R1_IMPROVED_SEMANTIC_MANUAL_PROMPT_TEMPLATE
    != ORIGINAL_FULL_R1_PROMPT_TEMPLATE
)


# Prefix before semantic section must be exactly identical.
assert (
    R1_IMPROVED_SEMANTIC_MANUAL_PROMPT_TEMPLATE[
        :len(PROMPT_BEFORE_SEMANTIC)
    ]
    == PROMPT_BEFORE_SEMANTIC
)


# Everything following the FINAL COMBINED DECISION header must
# remain exactly identical.
improved_after_final = (
    R1_IMPROVED_SEMANTIC_MANUAL_PROMPT_TEMPLATE.split(
        FINAL_DECISION_SECTION_HEADER,
        1,
    )[1]
)


original_after_final = (
    ORIGINAL_FULL_R1_PROMPT_TEMPLATE.split(
        FINAL_DECISION_SECTION_HEADER,
        1,
    )[1]
)


assert improved_after_final == original_after_final


# Required model-facing placeholders must remain present.
REQUIRED_MANUAL_PROMPT_PLACEHOLDERS = [
    "{duration_seconds}",
    "{participant_A_speaks}",
    "{participant_B_speaks}",
    "{local_temporal_features}",
    "{global_shift_features}",
    "{semantic_summaries}",
]


for placeholder in REQUIRED_MANUAL_PROMPT_PLACEHOLDERS:
    assert (
        placeholder
        in R1_IMPROVED_SEMANTIC_MANUAL_PROMPT_TEMPLATE
    ), (
        f"Missing required placeholder: {placeholder}"
    )


# No filtered turns or overlap may reappear.
prompt_lower = (
    R1_IMPROVED_SEMANTIC_MANUAL_PROMPT_TEMPLATE.lower()
)


FORBIDDEN_MANUAL_PROMPT_MARKERS = [
    "filtered turns:",
    "{participant_a_turns}",
    "{participant_b_turns}",
    "turn format and backchannel filtering",
    "clean_overlap_seconds",
    "clean_overlap_percent",
    "filtered overlap",
]


for marker in FORBIDDEN_MANUAL_PROMPT_MARKERS:
    assert marker not in prompt_lower, (
        f"Forbidden content found: {marker}"
    )


# Verify the critical semantic-policy concepts.
REQUIRED_SEMANTIC_POLICY_MARKERS = [
    "specific but unrelated content is INCOMPATIBLE",
    "semantic_assessment = COMPATIBLE",
    "semantic_assessment = INCOMPATIBLE",
    "semantic_assessment = LIMITED",
    "same conversation",
    "plausible",
    "concrete",
    "generic",
    "insufficient",
]


for marker in REQUIRED_SEMANTIC_POLICY_MARKERS:
    assert (
        marker.lower()
        in prompt_lower
    ), (
        f"Missing semantic-policy marker: {marker}"
    )


# ============================================================
# DISPLAY THE EXACT CHANGE
# ============================================================

semantic_policy_diff = "\n".join(
    difflib.unified_diff(
        ORIGINAL_SEMANTIC_POLICY_BODY.splitlines(),
        IMPROVED_SEMANTIC_POLICY_BODY.splitlines(),
        fromfile="original_R1_semantic_policy",
        tofile="improved_R1_semantic_policy",
        lineterm="",
    )
)


print("=" * 100)
print("R1 IMPROVED SEMANTIC MANUAL PROMPT DEFINED")
print("=" * 100)

print(
    "Original complete prompt characters:",
    len(
        ORIGINAL_FULL_R1_PROMPT_TEMPLATE
    ),
)

print(
    "Improved complete prompt characters:",
    len(
        R1_IMPROVED_SEMANTIC_MANUAL_PROMPT_TEMPLATE
    ),
)

print(
    "Original semantic-policy characters:",
    len(
        ORIGINAL_SEMANTIC_POLICY_BODY
    ),
)

print(
    "Improved semantic-policy characters:",
    len(
        IMPROVED_SEMANTIC_POLICY_BODY
    ),
)

print(
    "All non-semantic prompt sections preserved exactly:",
    True,
)

print(
    "Filtered turns present:",
    "filtered turns:" in prompt_lower,
)

print(
    "Overlap present:",
    "filtered overlap" in prompt_lower,
)

print("\n" + "=" * 100)
print("SEMANTIC POLICY DIFF")
print("=" * 100)

print(
    semantic_policy_diff
)

R1 IMPROVED SEMANTIC MANUAL PROMPT DEFINED
Original complete prompt characters: 18363
Improved complete prompt characters: 24003
Original semantic-policy characters: 1872
Improved semantic-policy characters: 7512
All non-semantic prompt sections preserved exactly: True
Filtered turns present: False
Overlap present: False

SEMANTIC POLICY DIFF
--- original_R1_semantic_policy
+++ improved_R1_semantic_policy
@@ -3,14 +3,14 @@
 - Segment 0: 0 to 60 seconds
 - Segment 1: 60 to 120 seconds
 
-For each participant and segment, you receive:
-
-Coarse semantic information:
+For each participant and synchronized segment, you receive:
+
+COARSE SUMMARY:
 
 - speech_content_summary
 - apparent_topic
 
-Focused semantic information:
+FOCUSED SUMMARY:
 
 - detailed_speech_summary
 - main_topic
@@ -20,48 +20,217 @@
 - unclear_content
 - confidence
 
-The semantic summaries were independently generated and may be broad,
-imperfect, repetitive, or uncertain.
-
-Evaluate semantic compatibility rather th

In [ ]:
# STEP 4 — PREPARE THE EXACT MANUALLY DEFINED EXPERIMENT

R1_IMPROVED_SEMANTIC_CONFIG = (
    prepare_manual_improved_semantic_experiment(
        manual_prompt_template=(
            R1_IMPROVED_SEMANTIC_MANUAL_PROMPT_TEMPLATE
        ),
    )
)


IMPROVED SEMANTIC POLICY — CONFIGURATION READY
Experiment version: structured_r1_improved_semantic_policy_speaks_offsets_all_global_no_overlap
Cases: 400
Semantic input: coarse_and_focused
Filtered turns supplied: False
Overlap supplied: False
Prompt SHA256: cd0016a024d9edde530a49fc9e32e3f34730e302dfb533b491a41fb095e0675a
Example prompt characters: 29087
Prediction cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/structured_r1_improved_semantic_policy_speaks_offsets_all_global_no_overlap/predictions_cache.json
Existing cache: False


In [ ]:
# STEP 5 — REQUIRED EXACT PROMPT INSPECTION

R1_IMPROVED_SEMANTIC_INSPECTION = (
    inspect_manual_improved_semantic_prompt(
        R1_IMPROVED_SEMANTIC_CONFIG,
        case_index=0,
    )
)


MANUAL PROMPT INSPECTION — STRUCTURED R1 IMPROVED SEMANTIC POLICY
Inspection case ID: consolidation_lag_2sec_000
Gold label is intentionally NOT printed inside the model prompt.
Participant evidence supplied: speaks only
Local evidence supplied: complete offset distribution only
Global evidence supplied: all five global temporal features
Semantic evidence supplied: coarse + focused
Filtered turns supplied: False
Overlap supplied: False

EXACT MODEL-FACING PAYLOAD
{
  "analysis_duration_seconds": 120.0,
  "participant_A": {
    "speaks": true
  },
  "participant_B": {
    "speaks": true
  },
  "local_temporal_features": {
    "signed_strict_offsets_seconds": [
      3.16,
      4.28
    ],
    "num_signed_strict_offsets": 2,
    "offset_mean_seconds": 3.72,
    "offset_median_seconds": 3.72,
    "offset_max_seconds": 4.28,
    "offset_p75_seconds": 4.0,
    "offset_p90_seconds": 4.17,
    "num_offsets_above_1_5_seconds": 2,
    "percent_offsets_above_1_5_seconds": 100.0
  },
  "global_s

## Load Qwen2.5-Omni Thinker

Run this only after the two source prompts have been printed, the complete combined prompt
has been manually defined, and the exact rendered prompt has been inspected.


In [ ]:
import torch
from transformers import Qwen2_5OmniThinkerForConditionalGeneration, Qwen2_5OmniProcessor
from qwen_omni_utils import process_mm_info

MODEL_ID = "Qwen/Qwen2.5-Omni-7B"

model = Qwen2_5OmniThinkerForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto",
)

processor = Qwen2_5OmniProcessor.from_pretrained(MODEL_ID)

print("Loaded:", MODEL_ID)

config.json:   0%|          | 0.00/13.2k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/233k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1346 [00:00<?, ?it/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/832 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-Omni-7B


In [ ]:
# STEP 6 — RUN ALL 400 CASES
# Every completed case is checkpointed to Google Drive.

R1_IMPROVED_SEMANTIC_CACHE = (
    run_manual_improved_semantic_experiment(
        R1_IMPROVED_SEMANTIC_CONFIG,
        print_each_case_prompt=False,
    )
)


Created cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/structured_r1_improved_semantic_policy_speaks_offsets_all_global_no_overlap/predictions_cache.json


structured_r1_improved_semantic_policy_speaks_offsets_all_global_no_overlap:   0%|          | 0/400 [00:00<?, …


IMPROVED SEMANTIC POLICY — INFERENCE COMPLETE
Cached records: 400
Prediction cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/structured_r1_improved_semantic_policy_speaks_offsets_all_global_no_overlap/predictions_cache.json


In [ ]:
from pathlib import Path


# Add the missing prompt-diff path expected by
# evaluate_reasoning_experiment().
config_paths = (
    R1_IMPROVED_SEMANTIC_CONFIG[
        "paths"
    ]
)


if "prompt_diff" not in config_paths:

    prompt_template_path = Path(
        config_paths[
            "prompt_template"
        ]
    )

    config_paths[
        "prompt_diff"
    ] = (
        prompt_template_path.parent
        /
        "prompt_diff.txt"
    )


# Save the semantic-policy diff when it is available.
prompt_diff_path = Path(
    config_paths[
        "prompt_diff"
    ]
)


if (
    "semantic_policy_diff"
    in globals()
):

    prompt_diff_path.write_text(
        semantic_policy_diff,
        encoding="utf-8",
    )

else:

    prompt_diff_path.touch(
        exist_ok=True
    )


print(
    "Prompt diff path added:",
    config_paths[
        "prompt_diff"
    ],
)

print(
    "Prompt diff exists:",
    prompt_diff_path.exists(),
)

Prompt diff path added: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/structured_r1_improved_semantic_policy_speaks_offsets_all_global_no_overlap/prompt_diff.txt
Prompt diff exists: True


In [ ]:
# STEP 7 — EVALUATE

R1_IMPROVED_SEMANTIC_EVALUATION = (
    evaluate_reasoning_experiment(
        R1_IMPROVED_SEMANTIC_CONFIG
    )
)


Structured R1 — Improved Semantic Policy — Speaks + Offsets + All Global + Coarse/Focused — No Overlap — RESULTS
Total cases: 400
Valid predictions: 400
Invalid predictions: 0
Exact structured-schema rate: 1.0000
Accuracy: 0.8700
Balanced accuracy: 0.8600
ANOMALOUS precision: 0.9429
ANOMALOUS recall: 0.8800
ANOMALOUS F1: 0.9103
NORMAL recall / specificity: 0.8400
MCC: 0.6803
Matched source-group exact rate: 0.5300
Reasoning inconsistencies: 0

CONFUSION MATRIX


,Pred NORMAL,Pred ANOMALOUS
Gold NORMAL,84,16
Gold ANOMALOUS,36,264



CLASSIFICATION REPORT


,precision,recall,f1-score,support
NORMAL,0.700000,0.84,0.763636,100.00
ANOMALOUS,0.942857,0.88,0.910345,300.00
accuracy,0.870000,0.87,0.870000,0.87
macro avg,0.821429,0.86,0.836991,400.00
weighted avg,0.882143,0.87,0.873668,400.00



PER CASE FAMILY


...


PER EXACT CASE VARIANT


,case_variant,total_cases,valid_predictions,invalid_predictions,predicted_NORMAL,predicted_ANOMALOUS,correct_predictions,accuracy_on_valid,strict_accuracy_invalid_as_wrong,exact_schema_rate
0,lag_2sec,50,50,0,15,35,35,0.70,0.70,1.0
1,lag_3sec,50,50,0,11,39,39,0.78,0.78,1.0
2,normal,100,100,0,84,16,84,0.84,0.84,1.0
3,silent_partner,100,100,0,0,100,100,1.00,1.00,1.0
4,wrong_partner,100,100,0,10,90,90,0.90,0.90,1.0



ASSESSMENT DISTRIBUTIONS


,assessment_field,assessment_value,count
0,participation_assessment,VALID,300
1,participation_assessment,INVALID,100
2,local_temporal_assessment,ANOMALOUS,229
3,local_temporal_assessment,NORMAL,169
4,local_temporal_assessment,LIMITED,2
5,global_temporal_assessment,ANOMALOUS,246
6,global_temporal_assessment,NORMAL,152
7,global_temporal_assessment,LIMITED,2
8,temporal_assessment,ANOMALOUS,246
9,temporal_assessment,NORMAL,152



Saved cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/structured_r1_improved_semantic_policy_speaks_offsets_all_global_no_overlap/predictions_cache.json
Saved prompt: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/structured_r1_improved_semantic_policy_speaks_offsets_all_global_no_overlap/prompt_template.txt
Saved prompt diff: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/structured_r1_improved_semantic_policy_speaks_offsets_all_global_no_overlap/prompt_diff.txt
Saved predictions: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/structured_r1_improved_semantic_policy_speaks_offsets_all_global_no_overlap/predictions_all_400.csv
Saved reasoning assessments: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/structured_r1_improved_semantic_policy_speaks_offsets_all_global_no_overlap/reasoning_assessments.csv
Saved errors: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/

In [ ]:
import pandas as pd
from IPython.display import display


# ============================================================
# LOAD R1 — IMPROVED SEMANTIC POLICY RESULTS
# ============================================================

if "R1_IMPROVED_SEMANTIC_EVALUATION" in globals():

    r1_improved_semantic_df = (
        R1_IMPROVED_SEMANTIC_EVALUATION[
            "results_df"
        ].copy()
    )

elif "R1_IMPROVED_SEMANTIC_CONFIG" in globals():

    r1_improved_semantic_df = pd.read_csv(
        R1_IMPROVED_SEMANTIC_CONFIG[
            "paths"
        ][
            "predictions_csv"
        ]
    )

else:

    raise RuntimeError(
        "Run the R1_IMPROVED_SEMANTIC "
        "configuration and evaluation cells first."
    )


# ============================================================
# NORMALIZE TEXT FIELDS
# ============================================================

upper_columns = [
    "gold_label",
    "prediction",
    "participation_assessment",
    "local_temporal_assessment",
    "global_temporal_assessment",
    "temporal_assessment",
    "semantic_assessment",
    "decisive_dimension",
]


for column in upper_columns:

    if column in r1_improved_semantic_df.columns:

        r1_improved_semantic_df[column] = (
            r1_improved_semantic_df[column]
            .astype("string")
            .str.strip()
            .str.upper()
        )


for column in [
    "case_family",
    "case_variant",
]:

    if column in r1_improved_semantic_df.columns:

        r1_improved_semantic_df[column] = (
            r1_improved_semantic_df[column]
            .astype("string")
            .str.strip()
            .str.lower()
            .str.replace(
                r"[\s\-]+",
                "_",
                regex=True,
            )
        )


# ============================================================
# VALID PREDICTION AND CORRECTNESS
# ============================================================

if (
    "valid_prediction"
    not in r1_improved_semantic_df.columns
):

    r1_improved_semantic_df[
        "valid_prediction"
    ] = (
        r1_improved_semantic_df[
            "prediction"
        ].isin([
            "NORMAL",
            "ANOMALOUS",
        ])
    )


if "correct" not in r1_improved_semantic_df.columns:

    r1_improved_semantic_df[
        "correct"
    ] = (
        r1_improved_semantic_df[
            "valid_prediction"
        ]
        &
        (
            r1_improved_semantic_df[
                "gold_label"
            ]
            ==
            r1_improved_semantic_df[
                "prediction"
            ]
        )
    )


# ============================================================
# EXPECTED STRUCTURED OUTPUT VALUES
# ============================================================

R1_IMPROVED_SEMANTIC_ASSESSMENT_LEVELS = {

    "participation_assessment": [
        "VALID",
        "INVALID",
    ],

    "local_temporal_assessment": [
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    ],

    "global_temporal_assessment": [
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    ],

    "temporal_assessment": [
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    ],

    "semantic_assessment": [
        "COMPATIBLE",
        "INCOMPATIBLE",
        "LIMITED",
    ],

    "decisive_dimension": [
        "PARTICIPATION",
        "TEMPORAL",
        "SEMANTIC",
        "NONE",
    ],
}


# ============================================================
# FREQUENCY TABLE
# ============================================================

def r1_improved_semantic_frequency_table(
    subset,
    field,
    expected_values,
):

    if field not in subset.columns:

        return pd.DataFrame({
            "assessment_field": [field],
            "assessment_value": [
                "COLUMN_NOT_AVAILABLE"
            ],
            "count": [0],
            "percentage": [0.0],
        })


    values = (
        subset[field]
        .fillna("MISSING")
        .astype(str)
        .str.strip()
        .str.upper()
    )


    ordered_values = list(
        dict.fromkeys(
            list(expected_values)
            +
            ["MISSING"]
        )
    )


    unexpected_values = [
        value
        for value in values.unique().tolist()
        if value not in ordered_values
    ]


    ordered_values.extend(
        sorted(
            unexpected_values
        )
    )


    counts = (
        values
        .value_counts(
            dropna=False
        )
        .reindex(
            ordered_values,
            fill_value=0,
        )
    )


    table = pd.DataFrame({
        "assessment_field": field,
        "assessment_value": counts.index,
        "count": counts.values,
    })


    if len(subset) > 0:

        table["percentage"] = (
            100.0
            *
            table["count"]
            /
            len(subset)
        ).round(2)

    else:

        table["percentage"] = 0.0


    return table


# ============================================================
# SAFE CROSSTAB
# ============================================================

def display_r1_improved_semantic_crosstab(
    subset,
    row_field,
    column_field,
    title,
):

    print(
        f"\n{title}"
    )


    if (
        row_field not in subset.columns
        or
        column_field not in subset.columns
    ):

        print(
            "Required columns are unavailable."
        )

        return


    display(
        pd.crosstab(
            subset[
                row_field
            ].fillna(
                "MISSING"
            ),

            subset[
                column_field
            ].fillna(
                "MISSING"
            ),

            margins=True,
        )
    )


# ============================================================
# COMPLETE SUBSET INSPECTION
# ============================================================

def inspect_r1_improved_semantic_subset(
    subset,
    title,
):

    subset = subset.copy()


    print(
        "\n"
        +
        "=" * 100
    )

    print(
        title
    )

    print(
        "=" * 100
    )


    if subset.empty:

        print(
            "No cases found in this subset."
        )

        return subset


    overview = pd.DataFrame([{

        "cases": len(subset),

        "gold_NORMAL": int(
            (
                subset[
                    "gold_label"
                ]
                ==
                "NORMAL"
            ).sum()
        ),

        "gold_ANOMALOUS": int(
            (
                subset[
                    "gold_label"
                ]
                ==
                "ANOMALOUS"
            ).sum()
        ),

        "pred_NORMAL": int(
            (
                subset[
                    "prediction"
                ]
                ==
                "NORMAL"
            ).sum()
        ),

        "pred_ANOMALOUS": int(
            (
                subset[
                    "prediction"
                ]
                ==
                "ANOMALOUS"
            ).sum()
        ),

        "invalid_predictions": int(
            (
                ~subset[
                    "valid_prediction"
                ]
            ).sum()
        ),

        "correct": int(
            subset[
                "correct"
            ].sum()
        ),

        "accuracy_percent": round(
            100.0
            *
            subset[
                "correct"
            ].mean(),
            2,
        ),
    }])


    print(
        "\nSUBSET OVERVIEW"
    )

    display(
        overview
    )


    # --------------------------------------------------------
    # CASE VARIANT BREAKDOWN
    # --------------------------------------------------------

    if "case_variant" in subset.columns:

        print(
            "\nCASE VARIANT BREAKDOWN"
        )


        variant_counts = (
            subset[
                "case_variant"
            ]
            .fillna(
                "MISSING"
            )
            .value_counts()
            .rename_axis(
                "case_variant"
            )
            .reset_index(
                name="count"
            )
        )


        variant_counts[
            "percentage"
        ] = (
            100.0
            *
            variant_counts[
                "count"
            ]
            /
            len(subset)
        ).round(2)


        display(
            variant_counts
        )


    # --------------------------------------------------------
    # ASSESSMENT BREAKDOWN
    # --------------------------------------------------------

    print(
        "\nASSESSMENT BREAKDOWN"
    )


    assessment_table = pd.concat(
        [
            r1_improved_semantic_frequency_table(
                subset=subset,
                field=field,
                expected_values=expected_values,
            )

            for field, expected_values
            in (
                R1_IMPROVED_SEMANTIC_ASSESSMENT_LEVELS
                .items()
            )
        ],

        ignore_index=True,
    )


    display(
        assessment_table
    )


    # --------------------------------------------------------
    # CROSSTABS
    # --------------------------------------------------------

    display_r1_improved_semantic_crosstab(
        subset,
        "participation_assessment",
        "decisive_dimension",
        (
            "PARTICIPATION ASSESSMENT "
            "× DECISIVE DIMENSION"
        ),
    )


    display_r1_improved_semantic_crosstab(
        subset,
        "local_temporal_assessment",
        "global_temporal_assessment",
        (
            "LOCAL × GLOBAL "
            "TEMPORAL ASSESSMENT"
        ),
    )


    display_r1_improved_semantic_crosstab(
        subset,
        "temporal_assessment",
        "decisive_dimension",
        (
            "TEMPORAL ASSESSMENT "
            "× DECISIVE DIMENSION"
        ),
    )


    display_r1_improved_semantic_crosstab(
        subset,
        "semantic_assessment",
        "decisive_dimension",
        (
            "SEMANTIC ASSESSMENT "
            "× DECISIVE DIMENSION"
        ),
    )


    display_r1_improved_semantic_crosstab(
        subset,
        "temporal_assessment",
        "semantic_assessment",
        (
            "TEMPORAL × SEMANTIC ASSESSMENT"
        ),
    )


    return subset


# ============================================================
# FAMILY MASK
# ============================================================

def get_r1_improved_semantic_family_mask(
    dataframe,
    family,
):

    values = (
        dataframe[
            "case_family"
        ]
        .fillna("")
        .astype(str)
        .str.lower()
    )


    if family == "normal":

        return values.isin([
            "normal",
            "normals",
        ])


    if family == "lag":

        return (
            values.isin([
                "lag",
                "lags",
            ])
            |
            values.str.contains(
                r"(?:^|_)lag(?:$|_)",
                regex=True,
            )
        )


    if family == "wrong_partner":

        return (
            values.isin([
                "wrong_partner",
                "wrong_partners",
            ])
            |
            (
                values.str.contains(
                    "wrong",
                    regex=False,
                )
                &
                values.str.contains(
                    "partner",
                    regex=False,
                )
            )
        )


    if family == "silent_partner":

        return (
            values.isin([
                "silent_partner",
                "silent_partners",
            ])
            |
            (
                values.str.contains(
                    "silent",
                    regex=False,
                )
                &
                values.str.contains(
                    "partner",
                    regex=False,
                )
            )
        )


    raise ValueError(
        f"Unknown family: {family}"
    )


# ============================================================
# CREATE OUTCOME SUBSET
# ============================================================

def get_r1_improved_semantic_outcome_subset(
    family,
    outcome,
):

    family_mask = (
        get_r1_improved_semantic_family_mask(
            dataframe=(
                r1_improved_semantic_df
            ),
            family=family,
        )
    )


    if family == "normal":

        gold_label = "NORMAL"

        if outcome == "correct":

            prediction = "NORMAL"

        elif outcome == "missed":

            prediction = "ANOMALOUS"

        else:

            raise ValueError(
                "outcome must be "
                "'correct' or 'missed'."
            )

    else:

        gold_label = "ANOMALOUS"

        if outcome == "correct":

            prediction = "ANOMALOUS"

        elif outcome == "missed":

            prediction = "NORMAL"

        else:

            raise ValueError(
                "outcome must be "
                "'correct' or 'missed'."
            )


    return r1_improved_semantic_df[
        family_mask
        &
        (
            r1_improved_semantic_df[
                "gold_label"
            ]
            ==
            gold_label
        )
        &
        (
            r1_improved_semantic_df[
                "prediction"
            ]
            ==
            prediction
        )
    ].copy()


# ============================================================
# INITIAL CHECK
# ============================================================

print(
    "R1 Improved Semantic cases loaded:",
    len(
        r1_improved_semantic_df
    ),
)


print(
    "\nCASE FAMILY COUNTS"
)

display(
    r1_improved_semantic_df[
        "case_family"
    ]
    .fillna(
        "MISSING"
    )
    .value_counts()
    .rename_axis(
        "case_family"
    )
    .reset_index(
        name="count"
    )
)


print(
    "\nCASE FAMILY × PREDICTION"
)

display(
    pd.crosstab(
        r1_improved_semantic_df[
            "case_family"
        ],

        r1_improved_semantic_df[
            "prediction"
        ],

        margins=True,
    )
)


print(
    "\nCASE FAMILY × SEMANTIC ASSESSMENT"
)

display(
    pd.crosstab(
        r1_improved_semantic_df[
            "case_family"
        ],

        r1_improved_semantic_df[
            "semantic_assessment"
        ],

        margins=True,
    )
)

R1 Improved Semantic cases loaded: 400

CASE FAMILY COUNTS


,case_family,count
0,normal,100
1,wrong_partner,100
2,lag,100
3,silent_partner,100



CASE FAMILY × PREDICTION


prediction,ANOMALOUS,NORMAL,All
case_family,,,
lag,74,26,100
normal,16,84,100
silent_partner,100,0,100
wrong_partner,90,10,100
All,280,120,400



CASE FAMILY × SEMANTIC ASSESSMENT


semantic_assessment,COMPATIBLE,INCOMPATIBLE,LIMITED,All
case_family,,,,
lag,95,2,3,100
normal,92,4,4,100
silent_partner,19,70,11,100
wrong_partner,45,40,15,100
All,251,116,33,400


In [ ]:
# ============================================================
# LAG — CORRECTLY DETECTED
# ============================================================

r1_improved_lag_correct = (
    get_r1_improved_semantic_outcome_subset(
        family="lag",
        outcome="correct",
    )
)

inspect_r1_improved_semantic_subset(
    r1_improved_lag_correct,
    (
        "R1 IMPROVED SEMANTIC — LAG CASES "
        "CORRECTLY DETECTED AS ANOMALOUS"
    ),
)


# ============================================================
# LAG — MISSED
# ============================================================

r1_improved_lag_missed = (
    get_r1_improved_semantic_outcome_subset(
        family="lag",
        outcome="missed",
    )
)

inspect_r1_improved_semantic_subset(
    r1_improved_lag_missed,
    (
        "R1 IMPROVED SEMANTIC — LAG CASES "
        "NOT DETECTED (PREDICTED NORMAL)"
    ),
)


# ============================================================
# WRONG PARTNER — CORRECTLY DETECTED
# ============================================================

r1_improved_wrong_partner_correct = (
    get_r1_improved_semantic_outcome_subset(
        family="wrong_partner",
        outcome="correct",
    )
)

inspect_r1_improved_semantic_subset(
    r1_improved_wrong_partner_correct,
    (
        "R1 IMPROVED SEMANTIC — WRONG-PARTNER "
        "CASES CORRECTLY DETECTED AS ANOMALOUS"
    ),
)


# ============================================================
# WRONG PARTNER — MISSED
# ============================================================

r1_improved_wrong_partner_missed = (
    get_r1_improved_semantic_outcome_subset(
        family="wrong_partner",
        outcome="missed",
    )
)

inspect_r1_improved_semantic_subset(
    r1_improved_wrong_partner_missed,
    (
        "R1 IMPROVED SEMANTIC — WRONG-PARTNER "
        "CASES NOT DETECTED (PREDICTED NORMAL)"
    ),
)


# ============================================================
# NORMAL — CORRECTLY DETECTED
# ============================================================

r1_improved_normal_correct = (
    get_r1_improved_semantic_outcome_subset(
        family="normal",
        outcome="correct",
    )
)

inspect_r1_improved_semantic_subset(
    r1_improved_normal_correct,
    (
        "R1 IMPROVED SEMANTIC — NORMAL CASES "
        "CORRECTLY PREDICTED AS NORMAL"
    ),
)


# ============================================================
# NORMAL — FALSELY FLAGGED
# ============================================================

r1_improved_normal_missed = (
    get_r1_improved_semantic_outcome_subset(
        family="normal",
        outcome="missed",
    )
)

inspect_r1_improved_semantic_subset(
    r1_improved_normal_missed,
    (
        "R1 IMPROVED SEMANTIC — NORMAL CASES "
        "INCORRECTLY PREDICTED AS ANOMALOUS"
    ),
)


# ============================================================
# SILENT PARTNER — CORRECTLY DETECTED
# ============================================================

r1_improved_silent_partner_correct = (
    get_r1_improved_semantic_outcome_subset(
        family="silent_partner",
        outcome="correct",
    )
)

inspect_r1_improved_semantic_subset(
    r1_improved_silent_partner_correct,
    (
        "R1 IMPROVED SEMANTIC — SILENT-PARTNER "
        "CASES CORRECTLY DETECTED AS ANOMALOUS"
    ),
)


# ============================================================
# SILENT PARTNER — MISSED
# ============================================================

r1_improved_silent_partner_missed = (
    get_r1_improved_semantic_outcome_subset(
        family="silent_partner",
        outcome="missed",
    )
)

inspect_r1_improved_semantic_subset(
    r1_improved_silent_partner_missed,
    (
        "R1 IMPROVED SEMANTIC — SILENT-PARTNER "
        "CASES NOT DETECTED (PREDICTED NORMAL)"
    ),
)


R1 IMPROVED SEMANTIC — LAG CASES CORRECTLY DETECTED AS ANOMALOUS

SUBSET OVERVIEW


,cases,gold_NORMAL,gold_ANOMALOUS,pred_NORMAL,pred_ANOMALOUS,invalid_predictions,correct,accuracy_percent
0,74,0,74,0,74,0,74,100.0



CASE VARIANT BREAKDOWN


,case_variant,count,percentage
0,lag_3sec,39,52.7
1,lag_2sec,35,47.3



ASSESSMENT BREAKDOWN


...

decisive_dimension,PARTICIPATION,SEMANTIC,TEMPORAL,All
semantic_assessment,,,,
COMPATIBLE,0,0,19,19
INCOMPATIBLE,4,19,47,70
LIMITED,11,0,0,11
All,15,19,66,100



TEMPORAL × SEMANTIC ASSESSMENT


semantic_assessment,COMPATIBLE,INCOMPATIBLE,LIMITED,All
temporal_assessment,,,,
ANOMALOUS,19,70,11,100
All,19,70,11,100



R1 IMPROVED SEMANTIC — SILENT-PARTNER CASES NOT DETECTED (PREDICTED NORMAL)
No cases found in this subset.


,case_id,source_group_id,case_family,case_variant,gold_label,prediction,participation_assessment,local_temporal_assessment,global_temporal_assessment,temporal_assessment,...,input_token_count,elapsed_seconds,raw_output,generation_error,valid_prediction,correct,normal_with_invalid_participation,normal_with_anomalous_temporal,normal_with_incompatible_semantics,reasoning_inconsistency


In [ ]:

import pandas as pd
from IPython.display import display


if "R1_IMPROVED_SEMANTIC_EVALUATION" in globals():
    r1s_df = (
        R1_IMPROVED_SEMANTIC_EVALUATION[
            "results_df"
        ].copy()
    )
elif "R1_IMPROVED_SEMANTIC_CONFIG" in globals():
    r1s_df = pd.read_csv(
        R1_IMPROVED_SEMANTIC_CONFIG[
            "paths"
        ][
            "predictions_csv"
        ]
    )
else:
    raise RuntimeError(
        "Run the improved semantic-policy evaluation first."
    )


for column in [
    "gold_label",
    "prediction",
    "participation_assessment",
    "local_temporal_assessment",
    "global_temporal_assessment",
    "temporal_assessment",
    "semantic_assessment",
    "decisive_dimension",
]:
    if column in r1s_df.columns:
        r1s_df[column] = (
            r1s_df[column]
            .astype("string")
            .str.strip()
            .str.upper()
        )


for column in [
    "case_family",
    "case_variant",
]:
    if column in r1s_df.columns:
        r1s_df[column] = (
            r1s_df[column]
            .astype("string")
            .str.strip()
            .str.lower()
            .str.replace(
                r"[\s\-]+",
                "_",
                regex=True,
            )
        )


if "valid_prediction" not in r1s_df.columns:
    r1s_df["valid_prediction"] = (
        r1s_df["prediction"].isin([
            "NORMAL",
            "ANOMALOUS",
        ])
    )

if "correct" not in r1s_df.columns:
    r1s_df["correct"] = (
        r1s_df["valid_prediction"]
        &
        (
            r1s_df["gold_label"]
            ==
            r1s_df["prediction"]
        )
    )


R1S_ASSESSMENT_LEVELS = {
    "participation_assessment": [
        "VALID",
        "INVALID",
    ],
    "local_temporal_assessment": [
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    ],
    "global_temporal_assessment": [
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    ],
    "temporal_assessment": [
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    ],
    "semantic_assessment": [
        "COMPATIBLE",
        "INCOMPATIBLE",
        "LIMITED",
    ],
    "decisive_dimension": [
        "PARTICIPATION",
        "TEMPORAL",
        "SEMANTIC",
        "NONE",
    ],
}


def r1s_frequency_table(
    subset,
    field,
    expected_values,
):
    if field not in subset.columns:
        return pd.DataFrame({
            "assessment_field": [field],
            "assessment_value": ["COLUMN_NOT_AVAILABLE"],
            "count": [0],
            "percentage": [0.0],
        })

    values = (
        subset[field]
        .fillna("MISSING")
        .astype(str)
        .str.strip()
        .str.upper()
    )

    ordered_values = list(
        dict.fromkeys(
            list(expected_values)
            + ["MISSING"]
        )
    )

    ordered_values.extend(
        sorted(
            value
            for value in values.unique().tolist()
            if value not in ordered_values
        )
    )

    counts = (
        values
        .value_counts(dropna=False)
        .reindex(
            ordered_values,
            fill_value=0,
        )
    )

    table = pd.DataFrame({
        "assessment_field": field,
        "assessment_value": counts.index,
        "count": counts.values,
    })

    table["percentage"] = (
        100.0
        * table["count"]
        / len(subset)
        if len(subset)
        else 0.0
    )

    table["percentage"] = (
        table["percentage"]
        .round(2)
    )

    return table


def get_r1s_family_mask(
    dataframe,
    family,
):
    values = (
        dataframe[
            "case_family"
        ]
        .fillna("")
        .astype(str)
        .str.lower()
    )

    if family == "normal":
        return values.eq("normal")

    if family == "lag":
        return (
            values.eq("lag")
            |
            values.str.contains(
                r"(?:^|_)lag(?:$|_)",
                regex=True,
            )
        )

    if family == "wrong_partner":
        return (
            values.eq("wrong_partner")
            |
            (
                values.str.contains(
                    "wrong",
                    regex=False,
                )
                &
                values.str.contains(
                    "partner",
                    regex=False,
                )
            )
        )

    if family == "silent_partner":
        return (
            values.eq("silent_partner")
            |
            (
                values.str.contains(
                    "silent",
                    regex=False,
                )
                &
                values.str.contains(
                    "partner",
                    regex=False,
                )
            )
        )

    raise ValueError(
        f"Unknown family: {family}"
    )


def get_r1s_outcome_subset(
    family,
    outcome,
):
    family_mask = (
        get_r1s_family_mask(
            r1s_df,
            family,
        )
    )

    if family == "normal":
        target_prediction = (
            "NORMAL"
            if outcome == "correct"
            else "ANOMALOUS"
        )
        gold_label = "NORMAL"
    else:
        target_prediction = (
            "ANOMALOUS"
            if outcome == "correct"
            else "NORMAL"
        )
        gold_label = "ANOMALOUS"

    return r1s_df[
        family_mask
        &
        (
            r1s_df[
                "gold_label"
            ]
            == gold_label
        )
        &
        (
            r1s_df[
                "prediction"
            ]
            == target_prediction
        )
    ].copy()


def inspect_r1s_subset(
    subset,
    title,
    *,
    show_cases=True,
):
    subset = subset.copy()

    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

    if subset.empty:
        print("No cases found.")
        return subset

    overview = pd.DataFrame([{
        "cases": len(subset),
        "pred_NORMAL": int(
            (
                subset[
                    "prediction"
                ]
                == "NORMAL"
            ).sum()
        ),
        "pred_ANOMALOUS": int(
            (
                subset[
                    "prediction"
                ]
                == "ANOMALOUS"
            ).sum()
        ),
        "correct": int(
            subset[
                "correct"
            ].sum()
        ),
        "accuracy_percent": round(
            100.0
            * subset[
                "correct"
            ].mean(),
            2,
        ),
    }])

    display(overview)

    assessment_table = pd.concat(
        [
            r1s_frequency_table(
                subset,
                field,
                expected_values,
            )
            for field, expected_values
            in R1S_ASSESSMENT_LEVELS.items()
        ],
        ignore_index=True,
    )

    print("\nASSESSMENT BREAKDOWN")
    display(assessment_table)

    for row_field, column_field, title_text in [
        (
            "local_temporal_assessment",
            "global_temporal_assessment",
            "LOCAL × GLOBAL TEMPORAL",
        ),
        (
            "temporal_assessment",
            "semantic_assessment",
            "TEMPORAL × SEMANTIC",
        ),
        (
            "semantic_assessment",
            "decisive_dimension",
            "SEMANTIC × DECISIVE DIMENSION",
        ),
        (
            "temporal_assessment",
            "decisive_dimension",
            "TEMPORAL × DECISIVE DIMENSION",
        ),
    ]:
        print("\n" + title_text)
        display(
            pd.crosstab(
                subset[
                    row_field
                ].fillna(
                    "MISSING"
                ),
                subset[
                    column_field
                ].fillna(
                    "MISSING"
                ),
                margins=True,
            )
        )

    if show_cases:
        preferred_columns = [
            "case_id",
            "case_family",
            "case_variant",
            "gold_label",
            "prediction",
            "participation_assessment",
            "local_temporal_assessment",
            "global_temporal_assessment",
            "temporal_assessment",
            "semantic_assessment",
            "decisive_dimension",
            "correct",
        ]

        available_columns = [
            column
            for column in preferred_columns
            if column in subset.columns
        ]

        display(
            subset[
                available_columns
            ]
            .sort_values(
                by=[
                    column
                    for column in [
                        "case_variant",
                        "case_id",
                    ]
                    if column
                    in available_columns
                ]
            )
            .reset_index(
                drop=True
            )
        )

    return subset


print(
    "Improved semantic-policy cases loaded:",
    len(r1s_df),
)

print("\nCASE FAMILY × PREDICTION")
display(
    pd.crosstab(
        r1s_df[
            "case_family"
        ],
        r1s_df[
            "prediction"
        ],
        margins=True,
    )
)

print("\nCASE FAMILY × SEMANTIC ASSESSMENT")
display(
    pd.crosstab(
        r1s_df[
            "case_family"
        ],
        r1s_df[
            "semantic_assessment"
        ],
        margins=True,
    )
)


Improved semantic-policy cases loaded: 400

CASE FAMILY × PREDICTION


prediction,ANOMALOUS,NORMAL,All
case_family,,,
lag,74,26,100
normal,16,84,100
silent_partner,100,0,100
wrong_partner,90,10,100
All,280,120,400



CASE FAMILY × SEMANTIC ASSESSMENT


semantic_assessment,COMPATIBLE,INCOMPATIBLE,LIMITED,All
case_family,,,,
lag,95,2,3,100
normal,92,4,4,100
silent_partner,19,70,11,100
wrong_partner,45,40,15,100
All,251,116,33,400


In [ ]:

for family, family_title in [
    ("normal", "NORMAL"),
    ("lag", "LAG"),
    ("wrong_partner", "WRONG PARTNER"),
    ("silent_partner", "SILENT PARTNER"),
]:
    for outcome, outcome_title in [
        ("correct", "CORRECT"),
        ("missed", "MISCLASSIFIED"),
    ]:
        subset = get_r1s_outcome_subset(
            family,
            outcome,
        )

        inspect_r1s_subset(
            subset,
            (
                "STRUCTURED R1 IMPROVED SEMANTIC POLICY — "
                f"{family_title} — {outcome_title}"
            ),
        )



STRUCTURED R1 IMPROVED SEMANTIC POLICY — NORMAL — CORRECT


,cases,pred_NORMAL,pred_ANOMALOUS,correct,accuracy_percent
0,84,84,0,84,100.0



ASSESSMENT BREAKDOWN


,assessment_field,assessment_value,count,percentage
0,participation_assessment,VALID,84,100.00
1,participation_assessment,INVALID,0,0.00
2,participation_assessment,MISSING,0,0.00
3,local_temporal_assessment,NORMAL,83,98.81
4,local_temporal_assessment,ANOMALOUS,0,0.00
5,local_temporal_assessment,LIMITED,1,1.19
6,local_temporal_assessment,MISSING,0,0.00
7,global_temporal_assessment,NORMAL,83,98.81
8,global_temporal_assessment,ANOMALOUS,0,0.00
9,global_temporal_assessment,LIMITED,1,1.19



LOCAL × GLOBAL TEMPORAL


...

decisive_dimension,PARTICIPATION,SEMANTIC,TEMPORAL,All
semantic_assessment,,,,
COMPATIBLE,0,0,19,19
INCOMPATIBLE,4,19,47,70
LIMITED,11,0,0,11
All,15,19,66,100



TEMPORAL × DECISIVE DIMENSION


decisive_dimension,PARTICIPATION,SEMANTIC,TEMPORAL,All
temporal_assessment,,,,
ANOMALOUS,15,19,66,100
All,15,19,66,100


,case_id,case_family,case_variant,gold_label,prediction,participation_assessment,local_temporal_assessment,global_temporal_assessment,temporal_assessment,semantic_assessment,decisive_dimension,correct
0,consolidation_silent_partner_000,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,INCOMPATIBLE,SEMANTIC,True
1,consolidation_silent_partner_001,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,INCOMPATIBLE,SEMANTIC,True
2,consolidation_silent_partner_002,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,INCOMPATIBLE,SEMANTIC,True
3,consolidation_silent_partner_003,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,INCOMPATIBLE,TEMPORAL,True
4,consolidation_silent_partner_004,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,COMPATIBLE,TEMPORAL,True
...,...,...,...,...,...,...,...,...,...,...,...,...
95,consolidation_silent_partner_095,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,INCOMPATIBLE,TEMPORAL,True
96,consolidation_silent_partner_096,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,INCOMPATIBLE,TEMPORAL,True
97,consolidation_silent_partner_097,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,COMPATIBLE,TEMPORAL,True
98,consolidation_silent_partner_098,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,COMPATIBLE,TEMPORAL,True



STRUCTURED R1 IMPROVED SEMANTIC POLICY — SILENT PARTNER — MISCLASSIFIED
No cases found.


# Optional: disconnect the Colab runtime

Run this only after inference/evaluation has finished or after you intentionally stop it.
Every completed case is checkpointed in Google Drive.


In [ ]:
from google.colab import runtime

runtime.unassign()
